# VisDrone PrivacyRisk ? Continuation t? Cell 6

Notebook n?y ti?p t?c pipeline t? k?t qu? ?? l?u ? Cell 6: ByteTrack, 21 features, Re-ID attack, g?n nh?n r?i ro, XGBoost v? export.

Y?u c?u: ?? c? `/kaggle/working/artifacts`, `/kaggle/working/runs`, `sequence_index.csv`, `data_person.yaml` v? checkpoint/model t? notebook tr??c.


In [1]:
# Kaggle bootstrap: keep notebook runnable when helper .py files were not uploaded.
import json
from pathlib import Path
import numpy as np
import pandas as pd
FEATURES = ("conf", "conf_std5", "conf_delta", "track_age", "hit_ratio", "time_since_update", "track_state", "min_side_px", "aspect_dev", "dist_edge", "is_trunc", "speed_norm", "pred_uncert", "lap_var_log", "brightness", "motion_blur_ratio", "nearest_dist_norm", "n_neighbors_r2_norm", "overlap_max", "frame_det_delta", "n_det_expected_ratio")
def add_validation_split(index, seed=42, validation_fraction=0.20):
    result = index.copy()
    if "val" in set(result["split"].astype(str)):
        return result
    sequences = sorted(result.loc[result["split"].eq("train"), "sequence"].unique())
    if len(sequences) < 2:
        raise ValueError("at least two train sequences are required")
    rng = np.random.default_rng(seed); rng.shuffle(sequences)
    chosen = set(sequences[:max(1, int(round(len(sequences) * validation_fraction)))])
    result.loc[result["split"].eq("train") & result["sequence"].isin(chosen), "split"] = "val"
    return result
def repair_tracker_features(df):
    result = df.copy()
    result["time_since_update"] = pd.to_numeric(result["time_since_update"], errors="coerce").clip(lower=0).fillna(0)
    result["hit_ratio"] = pd.to_numeric(result["hit_ratio"], errors="coerce").clip(0, 1)
    result["track_state"] = pd.to_numeric(result["track_state"], errors="coerce").fillna(2).clip(0, 2)
    return result
def save_model_metadata(model_path, feature_names=FEATURES):
    if tuple(feature_names) != tuple(FEATURES): raise ValueError("feature order mismatch")
    path = Path(model_path).with_suffix(Path(model_path).suffix + ".metadata.json")
    path.write_text(json.dumps({"feature_names": list(FEATURES), "feature_count": 21}, indent=2), encoding="utf-8")
    return path
def load_model_metadata(model_path):
    path = Path(model_path).with_suffix(Path(model_path).suffix + ".metadata.json")
    data = json.loads(path.read_text(encoding="utf-8"))
    if tuple(data.get("feature_names", ())) != tuple(FEATURES): raise ValueError("feature metadata mismatch")
    return tuple(FEATURES)


In [2]:
# ── CELL 1/13 · Setup + kiểm tra phần cứng ─────────────────
# Chỉ cài gói còn THIẾU. Không dùng -U để tránh pip gỡ bản có sẵn rồi tải lỗi
# (Kaggle: pip upgrade = uninstall trước, mất mạng giữa chừng là mất luôn gói).

import importlib.util
import subprocess
import sys

REQUIRED = {
    'ultralytics': 'ultralytics',
    'xgboost': 'xgboost',
    'shap': 'shap',
    'sklearn': 'scikit-learn',
    'pandas': 'pandas',
    'yaml': 'pyyaml',
    'onnx': 'onnx',
    'onnxruntime': 'onnxruntime',
    'matplotlib': 'matplotlib',
    'cv2': 'opencv-python-headless',
    'torch': 'torch',
}

_missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if _missing:
    print('gói còn thiếu:', _missing)
    _r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *_missing],
                        capture_output=True, text=True)
    if _r.returncode != 0:
        raise RuntimeError(
            'pip install thất bại. Bật Kaggle Settings → Internet: On rồi chạy lại cell này.\n'
            + (_r.stderr or '')[-1500:]
        )
else:
    print('tất cả gói đã có sẵn, không cần cài.')

# torchreid là tuỳ chọn (Re-ID dùng OSNet). Thiếu thì cell 9 tự fallback ResNet50.
if importlib.util.find_spec('torchreid') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torchreid'],
                   capture_output=True, text=True)

import os
for _k in ('OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[_k] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['CUDA_MODULE_LOADING'] = 'LAZY'

import csv
import ctypes
import gc
import json
import math
import random
import re
import shutil
import sys
import time
import warnings
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import ultralytics
from ultralytics import YOLO

# glibc (Kaggle = Linux): giới hạn arena + cho phép trả heap đã free về OS.
# Dùng kèm malloc_trim(0) trong Cell 7 để RSS không phình dần.
_LIBC = None
try:
    _LIBC = ctypes.CDLL('libc.so.6')
    _LIBC.mallopt(-8, 2)   # M_ARENA_MAX = 2
except Exception:
    _LIBC = None

warnings.filterwarnings('ignore')
torch.set_num_threads(1)
torch.backends.cudnn.benchmark = True

print('python     :', sys.version.split()[0])
print('ultralytics:', ultralytics.__version__)
print('torch      :', torch.__version__, '| cuda:', torch.cuda.is_available())

n_gpu = torch.cuda.device_count()
print('GPU count  :', n_gpu)
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name} | {p.total_memory / 2**30:.1f} GiB')

try:
    import psutil
    vm = psutil.virtual_memory()
    print(f'RAM        : {vm.total / 2**30:.1f} GiB total | {vm.available / 2**30:.1f} GiB available')
except ImportError:
    pass

for _p in ('/kaggle/working', '/tmp', '/kaggle/input'):
    if os.path.isdir(_p):
        u = shutil.disk_usage(_p)
        print(f'disk {_p:<16}: {u.free / 2**30:.1f} GiB free / {u.total / 2**30:.1f} GiB')

if n_gpu < 2:
    print('CẢNH BÁO: chỉ thấy', n_gpu, 'GPU. Vẫn chạy được nhưng chậm hơn.')
assert torch.cuda.is_available(), 'Bật GPU T4 x2 trong Kaggle Settings trước khi chạy.'


gói còn thiếu: ['ultralytics', 'onnxruntime']
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
python     : 3.13.15
ultralytics: 8.4.197
torch      : 2.11.0+cu128 | cuda: True
GPU count  : 2
  GPU 0: Tesla T4 | 14.6 GiB
  GPU 1: Tesla T4 | 14.6 GiB
RAM        : 31.3 GiB total | 29.6 GiB available
disk /kaggle/working : 19.5 GiB free / 19.5 GiB
disk /tmp            : 1048.7 GiB free / 8156.9 GiB
disk /kaggle/input   : 19.5 GiB free / 19.5 GiB


In [3]:
# ── CELL 2/13 · CONFIG ─────────────────────────────────────
# Chỉ sửa cell này.

from pathlib import Path

# ===== DỮ LIỆU =====
# Cách 1: 1 thư mục gốc, notebook tự chia sequence theo SPLIT_RATIOS.
DATA_ROOT = Path('/kaggle/input/visdrone-vid')

# Cách 2 (đang dùng): 3 thư mục riêng cho train/val/test.
USE_SEPARATE_DIRS = True
TRAIN_DIR = Path('/kaggle/input/datasets/mtu2006/visdrone-train/VisDrone2019-VID-train')
VAL_DIR   = Path('/kaggle/input/datasets/mtu2006/visdrone-val/VisDrone2019-VID-val')
TEST_DIR  = Path('/kaggle/input/datasets/mtu2006/visdrone-test/VisDrone2019-VID-test-dev')

SPLIT_RATIOS = (4, 1, 2)
SPLIT_SEED = 42

# ===== THƯ MỤC =====
WORK = Path('/kaggle/working')
BUILD_DIR = WORK / 'privacy_build'
RUNS_DIR = WORK / 'runs'
ARTIFACT = WORK / 'artifacts'

SEQ_INDEX = WORK / 'sequence_index.csv'
GT_CSV = WORK / 'gt_boxes.csv'
DATA_YAML = WORK / 'data_person.yaml'
LAYOUT_JSON = WORK / 'layout.json'

TRACKS_CSV = ARTIFACT / 'tracks_frame.csv'
FEATURES_CSV = ARTIFACT / 'features_frame.csv'
SAMPLES_CSV = ARTIFACT / 'samples.csv'
ATTACK_CSV = ARTIFACT / 'attack_results.csv'
TAU_JSON = ARTIFACT / 'tau.json'
RISK_CSV = ARTIFACT / 'risk_dataset.csv'
SCHEMA_JSON = ARTIFACT / 'feature_schema.json'
RISK_MODEL_JSON = ARTIFACT / 'risk_model.json'
RISK_OOF_CSV = ARTIFACT / 'risk_oof.csv'
RISK_METRICS_JSON = ARTIFACT / 'risk_metrics.json'
DET_METRICS_JSON = ARTIFACT / 'det_metrics.json'

# ===== DETECTOR =====
MODEL_NAME = 'yolo26s.pt'
FALLBACK = ['yolo11s.pt', 'yolo26n.pt', 'yolo11n.pt']
PERSON_CATS = (1, 2)            # VisDrone: 1=pedestrian, 2=people
IMGSZ = 1280
BATCH = 16
EPOCHS = 80
WORKERS = 4
DEVICE = '0,1'
DEVICE_EVAL = 0
SEED = 42
PATIENCE = 15
RUN_NAME = 'yolo26s_visdrone_recall_t4x2'
MAX_DET = 1000

AUG = dict(
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=5.0, translate=0.15, scale=0.7, shear=2.0, perspective=0.0005,
    fliplr=0.5, flipud=0.0,
    mosaic=0.8, mixup=0.05, copy_paste=0.0, erasing=0.2,
    close_mosaic=12, multi_scale=0.0,
)

# ===== ĐÁNH GIÁ DETECTOR =====
VAL_CONF = 0.001
VAL_IOU = 0.7
SWEEP_MAX_IMAGES = 1200
CONF_GRID = (0.01, 0.02, 0.03, 0.05, 0.08, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50)
IOU_GRID = (0.5, 0.6, 0.7)
MOT_IOU_TH = 0.5

# ===== TRACKING =====
TRACK_SPLITS = ('train', 'val', 'test')
TRACK_CONF = 0.05
TRACK_IOU = 0.5
TRACK_BUFFER = 45
TRACK_ID_IOU = 0.4
TRACK_RESUME = True      # bỏ qua sequence đã track xong trong *.partial.csv

# ===== TỰ PHỤC HỒI KHI GẦN HẾT RAM =====
TRACK_RAM_LIMIT_GB = None    # None = tự dò (cgroup Kaggle); đặt số để ép ngưỡng
TRACK_RAM_STOP_FRAC = 0.85   # tự dọn model khi dùng > 85% giới hạn RAM
TRACK_MEM_CHECK_EVERY = 25   # kiểm tra RAM mỗi N frame trong 1 sequence
TRACK_AUTO_RESTART_MAX = 6   # số lần tự dọn + chạy tiếp trước khi dừng hẳn

# ===== 21 ĐẶC TRƯNG =====
FEATURES = [
    'conf', 'conf_std5', 'conf_delta',
    'track_age', 'hit_ratio', 'time_since_update', 'track_state',
    'min_side_px', 'aspect_dev', 'dist_edge', 'is_trunc',
    'speed_norm', 'pred_uncert',
    'lap_var_log', 'brightness', 'motion_blur_ratio',
    'nearest_dist_norm', 'n_neighbors_r2_norm', 'overlap_max',
    'frame_det_delta', 'n_det_expected_ratio',
]
CONF_WINDOW = 5
PRED_UNCERT_WINDOW = 5
NEIGHBOR_RADIUS_FRAC = 0.10
NEIGHBOR_K = 10
EWMA_ALPHA = 0.3
TRUNC_MARGIN_PX = 2
CROP_RESIZE = 128
FRAME_STRIDE = 1

# ===== CỬA SỔ MẪU =====
WINDOW_SIZE = 60          # 0 = dùng cả track
WINDOW_STRIDE = 1        # sliding window: dự đoán ở từng frame
MIN_WINDOW_FRAMES = 20

# ===== ẨN DANH =====
ANON_METHOD = 'blur'      # 'blur' | 'pixelate'
ANON_KSIZE = 15
ANON_PIXEL_BLOCK = 8

# ===== RE-ID =====
REID_BACKBONE = 'auto'    # 'auto' | 'osnet_x1_0' | 'resnet50'
REID_INPUT = 256
REID_BATCH = 64
EMBED_STRIDE = 5
PROBE_STRIDE = 10
MAX_PROBE_CROPS = 8
MAX_GALLERY_IDS = 200
GALLERY_MIN_GAP = 30      # tăng số mẫu có gallery nhưng vẫn tránh cửa sổ lân cận
GALLERY_CROSS_SEQUENCE = True
MIN_GALLERY_IDS = 2

# ===== GÁN NHÃN =====
TAU_SELECT_SPLIT = 'val'
LABEL_MIN_MATCH_RATE = 0.5

# ===== MÔ HÌNH RỦI RO =====
XGB_PARAMS = dict(
    n_estimators=400, max_depth=4, learning_rate=0.05,
    subsample=0.9, colsample_bytree=0.9, min_child_weight=2,
    reg_lambda=1.0, objective='binary:logistic', eval_metric='aucpr',
    tree_method='hist', early_stopping_rounds=40, n_jobs=4, random_state=SEED,
)
N_FOLDS = 5
RISK_THRESHOLD = 0.5

# ===== EXPORT =====
EXPORT_ONNX = True
ONNX_OPSET = 17

for _p in (WORK, RUNS_DIR, ARTIFACT, BUILD_DIR):
    _p.mkdir(parents=True, exist_ok=True)

print('CONFIG OK |', MODEL_NAME, '| imgsz', IMGSZ, '| batch', BATCH, '| epochs', EPOCHS)
if USE_SEPARATE_DIRS:
    for _k, _v in (('train', TRAIN_DIR), ('val', VAL_DIR), ('test', TEST_DIR)):
        print(f'data {_k:<5}:', _v, '| exists:', _v.is_dir())
else:
    print('data  :', DATA_ROOT, '| exists:', DATA_ROOT.is_dir())
print('features:', len(FEATURES))
assert len(FEATURES) == 21, 'Phải có đúng 21 đặc trưng'


CONFIG OK | yolo26n.pt | imgsz 1024 | batch 32 | epochs 25
data train: /kaggle/input/datasets/mtu2006/visdrone-train/VisDrone2019-VID-train | exists: True
data val  : /kaggle/input/datasets/mtu2006/visdrone-val/VisDrone2019-VID-val | exists: True
data test : /kaggle/input/datasets/mtu2006/visdrone-test/VisDrone2019-VID-test-dev | exists: True
features: 21


In [4]:
# ── CELL 3/13 · Chuẩn hoá dataset → YOLO person ────────────
# Vào : DATA_ROOT (hoặc TRAIN_DIR/VAL_DIR/TEST_DIR)
# Ra  : BUILD_DIR/<split>/images+labels, data_person.yaml,
#       sequence_index.csv, gt_boxes.csv, layout.json

import yaml
from PIL import Image
from concurrent.futures import ThreadPoolExecutor

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
PSEUDO_MAX_GAP = 5
PSEUDO_IOU = 0.3
SPLITS = ('train', 'val', 'test')


def xyxy_iou(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), dtype=np.float32)
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    x1 = np.maximum(a[:, None, 0], b[None, :, 0])
    y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2])
    y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    union = area_a[:, None] + area_b[None, :] - inter
    return np.where(union > 0, inter / union, 0.0)


def find_layout(path):
    if not path.is_dir():
        return None, None
    queue = [(path, 0)]
    while queue:
        cur, depth = queue.pop(0)
        if (cur / 'images').is_dir() and (cur / 'labels').is_dir():
            return 'yolo', cur
        if (cur / 'annotations').is_dir() and (cur / 'sequences').is_dir():
            return 'raw', cur
        if depth < 2:
            queue += [(p, depth + 1) for p in sorted(cur.iterdir()) if p.is_dir()]
    return None, None


def image_size(path):
    try:
        with Image.open(path) as im:
            return im.size
    except Exception:
        arr = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
        return (arr.shape[1], arr.shape[0]) if arr is not None else (0, 0)


def parse_annotation(path):
    rows = defaultdict(list)
    for raw in path.read_text(encoding='utf-8', errors='ignore').splitlines():
        f = raw.split(',')
        if len(f) != 10:
            continue
        try:
            frame, tid = int(float(f[0])), int(float(f[1]))
            x, y, w, h = (float(v) for v in f[2:6])
            score, cat = int(float(f[6])), int(float(f[7]))
        except ValueError:
            continue
        if score == 0 or not (1 <= cat <= 11) or w <= 0 or h <= 0:
            continue
        rows[frame].append((cat, x, y, w, h, tid))
    return rows


def copy_many(pairs, workers=16):
    # Kaggle: /kaggle/input là mount FUSE (~1 MB/s/file, overhead mở file rất cao).
    # Symlink vẫn đọc qua mount đó nên phải copy thật về /kaggle/working.
    # Copy song song để bù chi phí mở file.
    todo = [(s, d) for s, d in pairs if not d.exists()]
    if not todo:
        return 0
    with ThreadPoolExecutor(max_workers=workers) as ex:
        list(ex.map(lambda p: shutil.copy2(p[0], p[1]), todo))
    return len(todo)


def write_label(path, lines):
    path.write_text('\n'.join(lines) + ('\n' if lines else ''), encoding='utf-8')


def parse_name(path):
    m = re.match(r'^(.*)_frame(\d+)$', path.stem)
    if m:
        return m.group(1), int(m.group(2))
    m = re.match(r'^(.+?)_(\d+)$', path.stem)
    if m:
        return m.group(1), int(m.group(2))
    return (path.parent.name or '__all__'), 0


def collect_sequences(kind, root):
    if kind == 'raw':
        return sorted(p.name for p in (root / 'sequences').iterdir() if p.is_dir())
    seqs = set()
    for p in (root / 'images').rglob('*'):
        if p.suffix.lower() in IMG_EXTS:
            seqs.add(parse_name(p)[0])
    return sorted(seqs)


def split_sequences(seqs, ratios, seed):
    arr = sorted(seqs)
    random.Random(seed).shuffle(arr)
    n = len(arr)
    a, b, c = ratios
    total = a + b + c
    n_tr = max(1, int(round(n * a / total)))
    n_va = max(1, int(round(n * b / total)))
    if n_tr + n_va >= n:
        n_tr = max(1, n - 2)
        n_va = 1
    return {'train': arr[:n_tr], 'val': arr[n_tr:n_tr + n_va], 'test': arr[n_tr + n_va:]}


def convert_raw(root, split, seq_filter, out, index_rows, gt_rows):
    img_dir, lbl_dir = out / 'images', out / 'labels'
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)
    ann = {p.stem: parse_annotation(p) for p in (root / 'annotations').glob('*.txt')}
    seqs = [p for p in sorted((root / 'sequences').iterdir()) if p.is_dir()]
    if seq_filter is not None:
        seqs = [p for p in seqs if p.name in seq_filter]

    n_img = 0
    pairs = []
    for seq in seqs:
        files = sorted(p for p in seq.iterdir() if p.suffix.lower() in IMG_EXTS)
        if not files:
            continue
        iw, ih = image_size(files[0])
        rows = ann.get(seq.name, {})
        for src in files:
            frame = int(src.stem) if src.stem.isdigit() else n_img
            flat = f'{seq.name}__{frame:07d}.jpg'
            pairs.append((src, img_dir / flat))
            lines = []
            for cat, x, y, w, h, tid in rows.get(frame, []):
                if cat not in PERSON_CATS:
                    continue
                lines.append(f'0 {(x + w / 2) / iw:.6f} {(y + h / 2) / ih:.6f} {w / iw:.6f} {h / ih:.6f}')
                gt_rows.append({'split': split, 'sequence': seq.name, 'frame': frame,
                                'target_id': int(tid), 'x1': x, 'y1': y,
                                'x2': x + w, 'y2': y + h, 'w': w, 'h': h})
            write_label(lbl_dir / f'{flat[:-4]}.txt', lines)
            index_rows.append({'split': split, 'sequence': seq.name, 'frame': frame,
                               'flat_name': flat, 'source_image': str(src),
                               'width': iw, 'height': ih})
            n_img += 1
    copy_many(pairs)
    return n_img


def convert_yolo(root, split, seq_filter, out, index_rows, gt_rows):
    img_dir, lbl_dir = out / 'images', out / 'labels'
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)
    sizes = {}
    n_img = 0
    pairs = []
    for src in sorted((root / 'images').rglob('*')):
        if src.suffix.lower() not in IMG_EXTS:
            continue
        seq, frame = parse_name(src)
        if seq_filter is not None and seq not in seq_filter:
            continue
        if seq not in sizes:
            sizes[seq] = image_size(src)
        iw, ih = sizes[seq]

        rel = src.relative_to(root / 'images')
        lab = (root / 'labels' / rel).with_suffix('.txt')
        lines = []
        if lab.is_file():
            for ln in lab.read_text(encoding='utf-8', errors='ignore').splitlines():
                f = ln.split()
                if len(f) != 5:
                    continue
                try:
                    cls = int(float(f[0]))
                    cx, cy, w, h = (float(v) for v in f[1:])
                except ValueError:
                    continue
                if cls not in (0, 1):
                    continue
                x1, y1 = (cx - w / 2) * iw, (cy - h / 2) * ih
                x2, y2 = (cx + w / 2) * iw, (cy + h / 2) * ih
                lines.append(f'0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}')
                gt_rows.append({'split': split, 'sequence': seq, 'frame': frame,
                                'target_id': -1, 'x1': x1, 'y1': y1,
                                'x2': x2, 'y2': y2, 'w': x2 - x1, 'h': y2 - y1})

        flat = f'{seq}__{frame:07d}.jpg'
        pairs.append((src, img_dir / flat))
        write_label(lbl_dir / f'{flat[:-4]}.txt', lines)
        index_rows.append({'split': split, 'sequence': seq, 'frame': frame,
                           'flat_name': flat, 'source_image': str(src),
                           'width': iw, 'height': ih})
        n_img += 1
    copy_many(pairs)
    return n_img


def assign_pseudo_ids(gt_rows):
    groups = defaultdict(list)
    for i, r in enumerate(gt_rows):
        if r['target_id'] < 0:
            groups[(r['split'], r['sequence'])].append(i)

    for key, idxs in groups.items():
        idxs.sort(key=lambda i: gt_rows[i]['frame'])
        by_frame = defaultdict(list)
        for i in idxs:
            by_frame[gt_rows[i]['frame']].append(i)
        active, next_id = {}, 0
        for frame in sorted(by_frame):
            for tid in list(active):
                if frame - active[tid][0] > PSEUDO_MAX_GAP:
                    del active[tid]
            cur = by_frame[frame]
            boxes = np.array([[gt_rows[i][k] for k in ('x1', 'y1', 'x2', 'y2')] for i in cur], float)
            used, used_j = set(), set()
            if active:
                tids = list(active)
                gb = np.array([active[t][1] for t in tids], float)
                iou = xyxy_iou(boxes, gb)
                for flat in np.argsort(-iou, axis=None):
                    a, b = divmod(int(flat), iou.shape[1])
                    if iou[a, b] < PSEUDO_IOU:
                        break
                    if a in used or b in used_j:
                        continue
                    used.add(a)
                    used_j.add(b)
                    gt_rows[cur[a]]['target_id'] = tids[b]
                    active[tids[b]] = (frame, boxes[a])
            for a, i in enumerate(cur):
                if a in used:
                    continue
                gt_rows[i]['target_id'] = next_id
                active[next_id] = (frame, boxes[a])
                next_id += 1
    return sum(1 for r in gt_rows if r['target_id'] < 0)



if BUILD_DIR.joinpath('train', 'images').is_dir() and GT_CSV.is_file():
    print('?? c? privacy_build v? gt_boxes.csv; b? qua chu?n h?a dataset.')
else:
    index_rows, gt_rows = [], []
    layout_info = {'mode': 'separate' if USE_SEPARATE_DIRS else 'auto_split', 'sources': {}}

    _free_gb = shutil.disk_usage(WORK).free / 2 ** 30
    print(f'đĩa trống /kaggle/working: {_free_gb:.1f} GB (cần ~12 GB để copy ảnh)')
    if _free_gb < 12:
        print('CẢNH BÁO: thiếu dung lượng. Dọn /kaggle/working hoặc giảm dữ liệu.')

    if USE_SEPARATE_DIRS:
        dirs = {'train': TRAIN_DIR, 'val': VAL_DIR, 'test': TEST_DIR}
        for split in SPLITS:
            kind, root = find_layout(dirs[split])
            if kind is None:
                raise FileNotFoundError(f'{split}: không thấy dữ liệu trong {dirs[split]}')
            layout_info['sources'][split] = {'kind': kind, 'root': str(root)}
            out = BUILD_DIR / split
            if kind == 'raw':
                n = convert_raw(root, split, None, out, index_rows, gt_rows)
            else:
                n = convert_yolo(root, split, None, out, index_rows, gt_rows)
            print(f'{split:<5} {n:>5} ảnh ({kind})')
    else:
        src = DATA_ROOT
        kind, root = find_layout(src)
        if kind is None:
            raise FileNotFoundError(
                f'Không thấy annotations/+sequences/ hoặc images/+labels/ trong {src}. '
                f'Kiểm tra lại DATA_ROOT hoặc bật USE_SEPARATE_DIRS.'
            )
        all_seqs = collect_sequences(kind, root)
        if len(all_seqs) < 3:
            raise ValueError(f'Chỉ có {len(all_seqs)} sequence, cần >=3 để tự chia. '
                             f'Đặt USE_SEPARATE_DIRS = True nếu đã có 3 thư mục riêng.')
        seqs = split_sequences(all_seqs, SPLIT_RATIOS, SPLIT_SEED)
        layout_info.update({'kind': kind, 'root': str(root), 'sequences': seqs})
        print('auto split:', {k: len(v) for k, v in seqs.items()})
        for split in SPLITS:
            out = BUILD_DIR / split
            if kind == 'raw':
                n = convert_raw(root, split, set(seqs[split]), out, index_rows, gt_rows)
            else:
                n = convert_yolo(root, split, set(seqs[split]), out, index_rows, gt_rows)
            print(f'{split:<5} {n:>5} ảnh | {seqs[split]}')

    n_pseudo = sum(1 for r in gt_rows if r['target_id'] < 0)
    assign_pseudo_ids(gt_rows)
    layout_info['pseudo_identity'] = n_pseudo > 0
    layout_info['pseudo_rows'] = n_pseudo

    df_idx = pd.DataFrame(index_rows).sort_values(['split', 'sequence', 'frame'])
    df_gt = pd.DataFrame(gt_rows).sort_values(['split', 'sequence', 'frame', 'target_id'])
    df_idx.to_csv(SEQ_INDEX, index=False)
    df_gt.to_csv(GT_CSV, index=False)
    LAYOUT_JSON.write_text(json.dumps(layout_info, indent=2, ensure_ascii=False), encoding='utf-8')

    DATA_YAML.write_text(yaml.safe_dump({
        'path': str(WORK),
        'train': str(BUILD_DIR / 'train' / 'images'),
        'val': str(BUILD_DIR / 'val' / 'images'),
        'test': str(BUILD_DIR / 'test' / 'images'),
        'nc': 1,
        'names': {0: 'person'},
    }, sort_keys=False, allow_unicode=True), encoding='utf-8')

    print('\nsequence_index.csv:', len(df_idx), 'dòng')
    print('gt_boxes.csv      :', len(df_gt), 'box |', df_gt.target_id.nunique(), 'identity')
    if n_pseudo:
        print(f'CẢNH BÁO: {n_pseudo} box không có target_id → dùng pseudo-ID (xấp xỉ).')
    print(DATA_YAML.read_text(encoding='utf-8'))


đĩa trống /kaggle/working: 19.5 GB (cần ~12 GB để copy ảnh)
train 24201 ảnh (raw)
val    2846 ảnh (raw)
test   6635 ảnh (raw)

sequence_index.csv: 33682 dòng
gt_boxes.csv      : 519196 box | 453 identity
path: /kaggle/working
train: /kaggle/working/privacy_build/train/images
val: /kaggle/working/privacy_build/val/images
test: /kaggle/working/privacy_build/test/images
nc: 1
names:
  0: person



In [5]:
# ?? CELL 6-RESTORE ? Kh?i ph?c k?t qu? v? model t? Cell 6 ?????
# Ch?y cell n?y sau setup/config, tr??c Cell 7.

WORK = Path('/kaggle/working')
ARTIFACT = WORK / 'artifacts'
RUNS_DIR = WORK / 'runs'
ARTIFACT.mkdir(parents=True, exist_ok=True)

CHECKPOINT_DIR = ARTIFACT / 'cell6_checkpoint'
CHECKPOINT_FILE = CHECKPOINT_DIR / 'detector_checkpoint.json'
DET_METRICS_JSON = ARTIFACT / 'det_metrics.json'

_checkpoint = {}
if CHECKPOINT_FILE.is_file():
    _checkpoint = json.loads(CHECKPOINT_FILE.read_text(encoding='utf-8'))
elif DET_METRICS_JSON.is_file():
    _checkpoint = {'det_metrics': json.loads(DET_METRICS_JSON.read_text(encoding='utf-8'))}

_threshold = _checkpoint.get('det_metrics', {}).get('best_threshold', {})
CONF_BEST = float(_checkpoint.get('CONF_BEST', _threshold.get('conf', TRACK_CONF)))
IOU_BEST = float(_checkpoint.get('IOU_BEST', _threshold.get('iou', TRACK_IOU)))
IMGSZ_USED = int(_checkpoint.get('IMGSZ_USED', IMGSZ))
BATCH_USED = int(_checkpoint.get('BATCH_USED', BATCH))

# Tự tìm ZIP/model đã upload trong /kaggle/input hoặc /kaggle/working.
# ZIP chỉ được giải nén nếu có yolo26s.pt, tránh giải nén nhầm dataset lớn.
_search_roots = [WORK, Path('/kaggle/input')]
_zip_candidates = []
for _root in _search_roots:
    if _root.is_dir():
        _zip_candidates.extend(_root.rglob('*.zip'))
for _zip_path in _zip_candidates:
    try:
        with zipfile.ZipFile(_zip_path) as _zf:
            _has_model = any(Path(n).name == 'yolo26s.pt' for n in _zf.namelist())
        if _has_model:
            print('Giải nén model ZIP:', _zip_path)
            with zipfile.ZipFile(_zip_path) as _zf:
                _zf.extractall(WORK)
    except (OSError, zipfile.BadZipFile) as _err:
        print('Bỏ qua ZIP không đọc được:', _zip_path, '|', str(_err)[:120])

_model_candidates = []
for _root in _search_roots:
    if _root.is_dir():
        _model_candidates.extend(_root.rglob('yolo26s.pt'))
_model_candidates = [p for p in _model_candidates if p.is_file()]
if _model_candidates:
    _preferred = [p for p in _model_candidates if 'weights' in p.parts]
    BEST_MODEL = (_preferred or _model_candidates)[0]
else:
    BEST_MODEL = RUNS_DIR / RUN_NAME / 'weights' / 'yolo26s.pt'
assert BEST_MODEL.is_file(), (
    'Không tìm thấy yolo26s.pt. Hãy upload ZIP kết quả có runs/*/weights/yolo26s.pt '
    'vào Kaggle Input hoặc /kaggle/working.'
)
# Đồng bộ các file pipeline nếu chúng nằm trong thư mục giải nén khác.
for _name in ('sequence_index.csv', 'data_person.yaml', 'gt_boxes.csv'):
    _current = globals().get(_name.upper().replace('.CSV', '').replace('.YAML', ''), None)
    _found = next((p for p in WORK.rglob(_name) if p.is_file()), None)
    if _found is not None:
        if _name == 'sequence_index.csv': SEQ_INDEX = _found
        elif _name == 'data_person.yaml': DATA_YAML = _found
        else: GT_CSV = _found

for _required in (SEQ_INDEX, DATA_YAML):
    assert _required.is_file(), f'Thi?u file ??u v?o: {_required}'

print('?? kh?i ph?c model:', BEST_MODEL)
print(f'IMGSZ={IMGSZ_USED}, BATCH={BATCH_USED}, CONF={CONF_BEST}, IOU={IOU_BEST}')

RESULT_ZIP = WORK / 'visdrone_privacy_results.zip'
def create_results_zip():
    roots = [ARTIFACT, RUNS_DIR]
    files = [DATA_YAML, SEQ_INDEX, WORK / 'bytetrack_person.yaml']
    with zipfile.ZipFile(RESULT_ZIP, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
        for root in roots:
            if root.is_dir():
                for path in root.rglob('*'):
                    if path.is_file() and path != RESULT_ZIP:
                        zf.write(path, path.relative_to(WORK))
        for path in files:
            if path.is_file() and path != RESULT_ZIP:
                zf.write(path, path.relative_to(WORK))
    return RESULT_ZIP


?? kh?i ph?c model: /kaggle/input/datasets/mtu2006/1234567/runs/yolo26n_visdrone_person_t4x2/weights/yolo26s.pt
IMGSZ=1024, BATCH=32, CONF=0.25, IOU=0.5


In [ ]:
# CELL 7/13 · ByteTrack -> tracks_frame.csv (chunked, auto-merge)
# Input : yolo26s.pt, sequence_index.csv
# Output: artifacts/tracks_frame.csv (merged from tracks_chunk_*.csv)
#         outputs/tracks_chunk_XXX.csv, outputs/chunk_state.json
#
# Chunking mechanism:
#   - Split sequences into small chunks (CHUNK_SIZE sequences/chunk)
#   - Each chunk saved to separate file -> no progress loss on RAM limit
#   - If RAM >= RAM_STOP_GIB: save current chunk, stop safely, save state
#   - Re-run cell to continue from next chunk (resume)
#   - After all chunks: merge all tracks_chunk_*.csv -> tracks_frame.csv

def _read_int(path):
    try:
        return int(Path(path).read_text().strip())
    except Exception:
        return None

def _cgroup_used_gib():
    cur = _read_int('/sys/fs/cgroup/memory.current') or _read_int(
        '/sys/fs/cgroup/memory/memory.usage_in_bytes')
    if cur is None:
        return None
    inactive = 0
    for stat in ('/sys/fs/cgroup/memory.stat', '/sys/fs/cgroup/memory/memory.stat'):
        try:
            for ln in Path(stat).read_text().splitlines():
                key, _, val = ln.partition(' ')
                if key in ('inactive_file', 'total_inactive_file'):
                    inactive = int(val)
                    break
            break
        except Exception:
            continue
    return max(cur - inactive, 0) / 2**30

def _cgroup_limit_gib():
    for path in ('/sys/fs/cgroup/memory.max', '/sys/fs/cgroup/memory/memory.limit_in_bytes'):
        v = _read_int(path)
        if v is not None and v < 2**60:
            return v / 2**30
    return None

def _mem_used_gib():
    used = _cgroup_used_gib()
    if used is not None:
        return used
    if psutil is not None:
        try:
            return psutil.Process().memory_info().rss / 2**30
        except Exception:
            return None
    return None

def _mem_snapshot():
    snap = {}
    if psutil is not None:
        try:
            snap['rss'] = psutil.Process().memory_info().rss / 2**30
            snap['avail'] = psutil.virtual_memory().available / 2**30
        except Exception:
            pass
    try:
        snap['objs'] = float(len(gc.get_objects()))
    except Exception:
        pass
    used = _mem_used_gib()
    if used is not None:
        snap['used'] = used
    try:
        snap['vram'] = torch.cuda.memory_allocated() / 2**30
    except Exception:
        pass
    return snap

def _vram_free_gib(idx):
    try:
        return torch.cuda.mem_get_info(idx)[0] / 2**30
    except Exception:
        return 0.0

def _trim_host_memory():
    gc.collect()
    if _LIBC is not None:
        try:
            _LIBC.malloc_trim(0)
        except Exception:
            pass
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def _same_device(pred, device):
    try:
        d = pred.device
        if str(device) == 'cpu':
            return d.type == 'cpu'
        return d.type == 'cuda' and int(d.index or 0) == int(device)
    except Exception:
        return False

def _reset_trackers(model, device):
    pred = getattr(model, 'predictor', None)
    if pred is None:
        return
    if not _same_device(pred, device):
        model.predictor = None
        _trim_host_memory()
        return
    if hasattr(pred, 'trackers'):
        del pred.trackers
    gc.collect()

class TrackMemoryStop(Exception):
    """Stop when memory cleanup does not bring RAM below threshold."""

# --- 1. Clean VRAM/RAM from Cell 5-6 ---
CHUNK_SIZE = int(globals().get('TRACK_CHUNK_SIZE', 5))  # sequences per chunk
CHUNK_DIR = Path('/kaggle/working/outputs')
CHUNK_DIR.mkdir(parents=True, exist_ok=True)
STATE_FILE = CHUNK_DIR / 'chunk_state.json'
CHUNK_PREFIX = 'tracks_chunk_'

# RAM threshold (reuse cell config if available)
CHUNK_RAM_LIMIT_GIB = float(globals().get('TRACK_RAM_LIMIT_GB') or 0.0) or _cgroup_limit_gib()
if not CHUNK_RAM_LIMIT_GIB and psutil is not None:
    try:
        CHUNK_RAM_LIMIT_GIB = psutil.virtual_memory().total / 2**30
    except Exception:
        CHUNK_RAM_LIMIT_GIB = None
CHUNK_RAM_STOP_FRAC = float(globals().get('TRACK_RAM_STOP_FRAC', 0.85))
CHUNK_RAM_STOP_GIB = CHUNK_RAM_LIMIT_GIB * CHUNK_RAM_STOP_FRAC if CHUNK_RAM_LIMIT_GIB else None
CHUNK_RAM_RESUME_GIB = (CHUNK_RAM_LIMIT_GIB * max(CHUNK_RAM_STOP_FRAC - 0.10, 0.50)) if CHUNK_RAM_LIMIT_GIB else None
print('Chunking:', CHUNK_SIZE, 'seq/chunk | RAM stop=', CHUNK_RAM_STOP_GIB, 'G' if CHUNK_RAM_STOP_GIB else 'RAM guard off')

# --- RAM/VRAM helpers ---
for _n in ('eval_model', 'sweep_model', 'track_model', 'model', 'exp',
           'trainer', 'results', 'val_results', 'sweep_results',
           'sweep_rows', 'det_results', 'metrics', 'fig', 'ax', 'predictor'):
    globals().pop(_n, None)
_trim_host_memory()
if torch.cuda.is_available():
    torch.cuda.ipc_collect()

# --- 2. Select least-loaded GPU on T4 x2 ---
TRACK_DEVICE = DEVICE_EVAL
TRACK_HALF = True
if torch.cuda.is_available():
    _cands = sorted(((_vram_free_gib(i), i) for i in range(torch.cuda.device_count())),
                    key=lambda x: x[1])
    print('VRAM free:', ' | '.join(f'GPU{i}={f:.1f}GiB' for f, i in _cands))
    _best_free, _best_idx = max(_cands)
    if _best_free >= 4.0:
        TRACK_DEVICE = _best_idx
    else:
        TRACK_DEVICE, TRACK_HALF = 'cpu', False
else:
    TRACK_DEVICE, TRACK_HALF = 'cpu', False

# --- 3. Resolution: keep train imgsz ---
TRACK_IMGSZ = int(globals().get('IMGSZ_USED', IMGSZ))
TRACK_PLANS = [(TRACK_IMGSZ, TRACK_DEVICE, TRACK_HALF)]

CONF_TRACK = max(float(globals().get('CONF_BEST', TRACK_CONF)), TRACK_CONF)
IOU_TRACK = float(globals().get('IOU_BEST', TRACK_IOU))
TRACK_RESUME = bool(globals().get('TRACK_RESUME', True))
print('tracking: imgsz=', TRACK_PLANS[0][0], '(train', TRACK_IMGSZ, ') conf=', CONF_TRACK,
      'iou=', IOU_TRACK, 'max_det=', MAX_DET, 'half=', TRACK_HALF, 'resume=', TRACK_RESUME)
print('Fixed imgsz:', TRACK_IMGSZ, '| device:', TRACK_DEVICE, '| half:', TRACK_HALF)

FIELDS = ['split', 'sequence', 'frame', 'track_id', 'class_id', 'confidence',
          'x1', 'y1', 'x2', 'y2', 'width', 'height']

_st = {'model': None, 'cb': {}, 'cb_base': {}, 'resets': 0, 'resets_total': 0}

def _hard_reset():
    """Release model + clean memory. True if RAM below resume threshold."""
    _st['model'] = None
    _st['cb'] = {}
    _st['cb_base'] = {}
    _trim_host_memory()
    if torch.cuda.is_available():
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass
    used = _mem_used_gib()
    if CHUNK_RAM_RESUME_GIB is None:
        return True
    return used is not None and used < CHUNK_RAM_RESUME_GIB

def _track_sequence(model, split, seq, pairs, imgsz, device, half):
    """Track 1 sequence. Return (rows, n_frames)."""

    frame_of = {str(p[1]): p[0] for p in pairs}
    size_of = {str(p[1]): (p[2], p[3]) for p in pairs}
    _reset_trackers(model, device)
    rows = []
    n = 0
    gen = None
    try:
        gen = model.track(
            source=[str(p[1]) for p in pairs], persist=True, tracker=str(TRACKER_YAML),
            conf=CONF_TRACK, iou=IOU_TRACK, imgsz=imgsz, max_det=MAX_DET,
            batch=1, half=half, device=device, stream=True, stream_buffer=False,
            vid_stride=1, verbose=False,
        )
        for res in gen:
            n += 1
            if MEM_EVERY and n % MEM_EVERY == 0 and _over_mem_chunk():
                raise TrackMemoryStop(
                    f'RAM {_mem_used_gib():.1f}G >= threshold {CHUNK_RAM_STOP_GIB:.1f}G '
                    f'at {split}/{seq} frame {n}')
            path = str(getattr(res, 'path', ''))
            frame = frame_of.get(path)
            if frame is None:
                frame = pairs[min(n, len(pairs) - 1)][0]
            iw, ih = size_of.get(path, (0, 0))
            b = res.boxes
            if b is None or len(b) == 0:
                del b, res
                continue
            xy = b.xyxy.cpu().numpy()
            cf = b.conf.cpu().numpy()
            cl = b.cls.int().cpu().numpy()
            ids = b.id.int().cpu().numpy() if b.id is not None else -np.ones(len(xy), int)
            for box, s, c, t in zip(xy, cf, cl, ids):
                rows.append((split, seq, frame, int(t), int(c), round(float(s), 6),
                             round(float(box[0]), 2), round(float(box[1]), 2),
                             round(float(box[2]), 2), round(float(box[3]), 2),
                             iw, ih))
            del b, xy, cf, cl, ids, res
    finally:
        if gen is not None:
            try:
                gen.close()
            except Exception:
                pass
        try:
            model.predictor = None
        except Exception:
            pass
        _trim_host_memory()
    return rows, n

def _over_mem_chunk():
    if CHUNK_RAM_STOP_GIB is None:
        return False
    used = _mem_used_gib()
    return used is not None and used >= CHUNK_RAM_STOP_GIB

def _write_track_config(final):
    TRACK_CONFIG_JSON = ARTIFACT / 'track_config.json'
    TRACK_CONFIG_JSON.write_text(json.dumps({
        'track_imgsz': TRACK_IMGSZ,
        'conf': CONF_TRACK, 'iou': IOU_TRACK, 'max_det': MAX_DET,
        'track_buffer': TRACK_BUFFER,
        'half': bool(TRACK_HALF), 'device': str(TRACK_DEVICE),
        'ladder': [{'imgsz': s, 'device': str(d), 'half': bool(h)} for s, d, h in TRACK_PLANS],
        'ram_limit_gib': CHUNK_RAM_LIMIT_GIB,
        'ram_stop_gib': CHUNK_RAM_STOP_GIB,
        'auto_resets': _st['resets_total'],
        'completed': bool(final),
        'chunk_dir': str(CHUNK_DIR),
    }, indent=2, ensure_ascii=False), encoding='utf-8')
    return TRACK_CONFIG_JSON

# --- Read chunk resume state ---
done_chunks = []
if STATE_FILE.is_file():
    try:
        _stt = json.loads(STATE_FILE.read_text(encoding='utf-8'))
        done_chunks = _stt.get('done_chunks', [])
        print('Resume chunk: already completed', len(done_chunks), 'chunks')
    except Exception as _e:
        print('Chunk state file corrupt, restarting from scratch:', _e)
        done_chunks = []

df_idx = pd.read_csv(SEQ_INDEX, dtype={'split': 'string', 'sequence': 'string',
                                       'frame': 'int32', 'width': 'int32',
                                       'height': 'int32'})
df_idx = (df_idx[df_idx['split'].isin(list(TRACK_SPLITS))]
          .sort_values(['split', 'sequence', 'frame'], kind='stable'))

all_groups = [(str(sp), str(sq), g)
              for (sp, sq), g in df_idx.groupby(['split', 'sequence'], sort=True, observed=True)]

already_done_seqs = set()
for _cf in sorted(CHUNK_DIR.glob(f'{CHUNK_PREFIX}*.csv')):
    try:
        _df_prev = pd.read_csv(_cf, usecols=['split', 'sequence'],
                              dtype={'split': 'string', 'sequence': 'string'})
        for _r in _df_prev.itertuples():
            already_done_seqs.add((str(_r.split), str(_r.sequence)))
    except Exception:
        pass

pending = [g for g in all_groups if (g[0], g[1]) not in already_done_seqs]
print('Total sequences:', len(all_groups), '| already tracked:', len(already_done_seqs),
      '| remaining:', len(pending), '| frames:', sum(len(g) for _s, _q, g in pending))

def _chunk_list(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i + size]

chunks = list(_chunk_list(pending, CHUNK_SIZE))
print('Split into', len(chunks), 'chunks (size', CHUNK_SIZE, 'sequences/chunk)')

def _save_chunk(chunk_idx, rows):
    chunk_csv = CHUNK_DIR / f'{CHUNK_PREFIX}{chunk_idx:03d}.csv'
    _new = not chunk_csv.is_file() or chunk_csv.stat().st_size == 0
    with open(chunk_csv, 'a', newline='', encoding='utf-8') as _f:
        _w = csv.writer(_f)
        if _new:
            _w.writerow(FIELDS)
        _w.writerows(rows)
        _f.flush()
        os.fsync(_f.fileno())
    print('  └─ Save chunk', chunk_idx, ':', len(rows), 'rows ->', chunk_csv.name)
    return chunk_csv

def _save_state(finished_chunks):
    STATE_FILE.write_text(json.dumps({
        'done_chunks': finished_chunks,
        'total_chunks': len(chunks),
        'chunk_size': CHUNK_SIZE,
    }, indent=2, ensure_ascii=False), encoding='utf-8')

# --- Main loop over chunks ---
used_plan = {}
plan_idx = 0
t0 = time.time()
n_done = 0
total_rows = 0
mem_stop = False
mem_stop_msg = ''
mem0 = _mem_snapshot()

for chunk_idx, chunk_groups in enumerate(chunks):
    if chunk_idx in done_chunks:
        continue

    # RAM check before chunk
    if _over_mem_chunk():
        if _st['resets'] < 6 and _hard_reset():
            _st['resets'] += 1
            _st['resets_total'] += 1
            print('  ↻ RAM threshold before chunk', chunk_idx, '-> cleaned model, continue')
        else:
            mem_stop = True
            mem_stop_msg = ('RAM', _mem_used_gib(), 'G >= threshold', CHUNK_RAM_STOP_GIB, 'G '
                            'before chunk', chunk_idx)
            break

    chunk_rows = []
    chunk_done = 0

    for split, seq, g in chunk_groups:
        pairs = [(int(r.frame), BUILD_DIR / split / 'images' / r.flat_name,
                  int(r.width), int(r.height)) for r in g.sort_values('frame').itertuples()]
        pairs = [p for p in pairs if p[1].exists()]
        if not pairs:
            print(f'  {split:<5} {seq:<24} skip (no image)')
            continue

        if _st['model'] is None:
            _st['model'] = YOLO(str(BEST_MODEL))
            _st['cb'] = getattr(_st['model'], 'callbacks', {})
            _st['cb_base'] = {k: len(v) for k, v in _st['cb'].items()}

        rows, n_seen, ok, last_err = None, 0, False, None
        stop_here, stop_msg = False, ''
        try:
            rows, n_seen = _track_sequence(_st['model'], split, seq, pairs, TRACK_IMGSZ, TRACK_DEVICE, TRACK_HALF)
            used_plan[f'{split}/{seq}'] = {'imgsz': TRACK_IMGSZ, 'device': str(TRACK_DEVICE), 'half': bool(TRACK_HALF)}
            ok = True
        except TrackMemoryStop as _ms:
            stop_here, stop_msg = True, str(_ms)

        if stop_here:
            if _st['resets'] < 6 and _hard_reset():
                _st['resets'] += 1
                _st['resets_total'] += 1
                print('  ↻', stop_msg, '-> cleaned model, retry this sequence')
                del rows, pairs, g
                continue
            mem_stop = True
            mem_stop_msg = stop_msg
            print('  ! Cannot lower RAM after cleanup -> stop at', split, '/', seq,
                  '(sequence NOT saved, will track next run)')
            del rows, pairs, g
            _trim_host_memory()
            break

        if not ok:
            raise RuntimeError(f'OOM at fixed imgsz={TRACK_IMGSZ} for {split}/{seq}: {last_err}')

        chunk_rows.extend(rows)
        chunk_done += 1
        n_done += 1
        total_rows += len(rows)
        _st['resets'] = 0

        # Trim stale callbacks
        for _k, _n_cb in _st['cb_base'].items():
            _lst = _st['cb'].get(_k)
            if _lst is not None and len(_lst) > _n_cb:
                del _lst[_n_cb:]

        del rows, pairs, g
        _trim_host_memory()

        _plan = used_plan[f'{split}/{seq}']
        _mem = _mem_snapshot()
        _used = _mem.get('used', float('nan'))
        _pct = f'{_used / CHUNK_RAM_LIMIT_GIB:.0%}' if (CHUNK_RAM_LIMIT_GIB and _used == _used) else '--'
        _eta = (time.time() - t0) / n_done * (len(pending) - n_done) / 60.0 if n_done else 0
        print(f'  {split:<5} {seq:<24} {n_seen:>4} frame | {total_rows:>7} box | '
              f'imgsz={_plan["imgsz"]:<4} | '
              f'RAM {_used:5.2f}G/{CHUNK_RAM_LIMIT_GIB or 0:4.1f}G ({_pct}) Δ{_used - mem0.get("used", 0):+5.2f} | '
              f'RSS {_mem.get("rss", float("nan")):5.2f}G | '
              f'objs {_mem.get("objs", 0) / 1000:6.0f}k | '
              f'VRAM {_mem.get("vram", float("nan")):4.2f}G | ETA {_eta:5.1f} min')

    # Save this chunk to file
    if chunk_rows:
        _save_chunk(chunk_idx, chunk_rows)

    # Mark chunk as done
    done_chunks.append(chunk_idx)
    _save_state(done_chunks)

    # Release memory after each chunk
    _st['model'] = None
    _trim_host_memory()

    if mem_stop:
        break

# --- End: safe stop or merge ---
if mem_stop:
    _left = len(pending) - n_done
    print('\n' + '=' * 78)
    print('SAFE STOP:', mem_stop_msg)
    print('Saved', n_done, 'sequences /', total_rows, 'rows into chunk files.')
    print('Remaining', _left, 'sequences not tracked. Re-run this cell to resume.')
    print('To free RAM completely: Restart session -> re-run Cell 7.')
    print('=' * 78)
    print('track_config.json ->', _write_track_config(False))
    _trim_host_memory()
    raise TrackMemoryStop(
        f'RAM still above threshold after {_st["resets_total"]} cleanups. '
        f'Saved {n_done} sequences into chunks -> re-run this cell to continue '
        f'{_left} remaining sequences.')

# --- Merge all chunks into tracks_frame.csv ---
chunk_files = sorted(CHUNK_DIR.glob(f'{CHUNK_PREFIX}*.csv'))
if not chunk_files:
    raise RuntimeError('No chunk files to merge. Possibly no chunk ran yet.')

print('\nMerging', len(chunk_files), 'chunk files ->', TRACKS_CSV.name, '...')
_DT = {'split': 'string', 'sequence': 'string', 'frame': 'int32', 'track_id': 'int32',
       'class_id': 'int8', 'confidence': 'float32', 'x1': 'float32', 'y1': 'float32',
       'x2': 'float32', 'y2': 'float32', 'width': 'int32', 'height': 'int32'}

_all_dfs = []
for _cf in chunk_files:
    _df = pd.read_csv(_cf, dtype=_DT)
    _all_dfs.append(_df)

df_all = pd.concat(_all_dfs, ignore_index=True)
df_all = df_all.sort_values(['split', 'sequence', 'frame', 'track_id'], kind='stable')
df_all.to_csv(TRACKS_CSV, index=False)

print('tracks_frame.csv:', len(df_all), 'rows |', df_all.track_id.nunique(), 'track_id')
print(df_all.groupby('split').agg(frames=('frame', 'nunique'),
                                  rows=('track_id', 'size')).to_string())

# Keep chunk files for audit (do not delete)

print('track_config.json ->', _write_track_config(True))

_trim_host_memory()


In [7]:
# ── CELL 8/13 · Trích 21 đặc trưng mỗi frame ───────────────
# Vào : tracks_frame.csv, sequence_index.csv, gt_boxes.csv
# Ra  : features_frame.csv, feature_schema.json

def crop_gray(img, box):
    if img is None:
        return None
    h, w = img.shape[:2]
    x1, y1, x2, y2 = box
    x1 = int(max(0, math.floor(x1)))
    y1 = int(max(0, math.floor(y1)))
    x2 = int(min(w, math.ceil(x2)))
    y2 = int(min(h, math.ceil(y2)))
    if x2 <= x1 or y2 <= y1:
        return None
    c = img[y1:y2, x1:x2]
    if max(c.shape[:2]) > CROP_RESIZE:
        s = CROP_RESIZE / max(c.shape[:2])
        c = cv2.resize(c, (max(1, int(c.shape[1] * s)), max(1, int(c.shape[0] * s))),
                       interpolation=cv2.INTER_AREA)
    return c


def lap_var(c):
    if c is None or c.size == 0:
        return 0.0
    return float(cv2.Laplacian(c, cv2.CV_64F).var())


# ── đọc tracks: Cell 7 có thể đã dừng an toàn -> dùng *.partial.csv ──
_TRK_SRC = TRACKS_CSV
if not _TRK_SRC.is_file():
    _TRK_ALT = TRACKS_CSV.with_name(TRACKS_CSV.stem + '.partial.csv')
    if not _TRK_ALT.is_file():
        raise FileNotFoundError(
            f'Không thấy {TRACKS_CSV.name} lẫn {_TRK_ALT.name}. '
            f'Chạy Cell 7/13 trước.')
    _TRK_SRC = _TRK_ALT
    print('!' * 78)
    print(f'CHƯA TRACK XONG: dùng {_TRK_ALT.name} (dữ liệu MỘT PHẦN).')
    print('Chạy lại Cell 7/13 để hoàn tất, rồi chạy lại các cell sau.')
    print('!' * 78)
df_trk = pd.read_csv(_TRK_SRC)
df_idx = add_validation_split(pd.read_csv(SEQ_INDEX), seed=SEED, validation_fraction=0.20)
df_idx.to_csv(SEQ_INDEX, index=False)
if GT_CSV.is_file():
    _gt = pd.read_csv(GT_CSV)
    _seq_split = df_idx[['sequence', 'split']].drop_duplicates()
    _gt = _gt.drop(columns=['split'], errors='ignore').merge(_seq_split, on='sequence', how='inner')
    _gt.to_csv(GT_CSV, index=False)
df_gt = pd.read_csv(GT_CSV)

meta = {(r.split, r.sequence, int(r.frame)): (int(r.width), int(r.height), r.flat_name)
        for r in df_idx.itertuples()}

train_gt = df_gt[df_gt.split == 'train']
r_ref = float(np.median(train_gt.w / train_gt.h)) if len(train_gt) else 0.4
print('r_ref (median w/h train):', round(r_ref, 4))

rows = []
for (split, seq), g in df_trk.groupby(['split', 'sequence'], sort=True):
    g = g.sort_values(['frame', 'track_id'])
    state = {}
    prev_n = None
    ewma = None

    for frame, fsub in g.groupby('frame', sort=True):
        frame = int(frame)
        iw, ih, flat = meta.get((split, seq, frame), (0, 0, None))
        if iw == 0 or flat is None:
            continue
        D = math.hypot(iw, ih)
        img = cv2.imread(str(BUILD_DIR / split / 'images' / flat), cv2.IMREAD_GRAYSCALE)

        boxes = fsub[['x1', 'y1', 'x2', 'y2']].to_numpy(float)
        N = len(boxes)
        cx = (boxes[:, 0] + boxes[:, 2]) / 2.0
        cy = (boxes[:, 1] + boxes[:, 3]) / 2.0
        centers = np.stack([cx, cy], 1)
        areas = (np.clip(boxes[:, 2] - boxes[:, 0], 1e-6, None) *
                 np.clip(boxes[:, 3] - boxes[:, 1], 1e-6, None))

        dx = centers[:, None, 0] - centers[None, :, 0]
        dy = centers[:, None, 1] - centers[None, :, 1]
        dist = np.hypot(dx, dy)
        np.fill_diagonal(dist, np.inf)
        nearest = np.clip(dist.min(1) / D, 0, 1) if N > 1 else np.ones(N)
        radius = NEIGHBOR_RADIUS_FRAC * D
        n_nb = np.clip((dist <= radius).sum(1) / NEIGHBOR_K, 0, 1) if N > 1 else np.zeros(N)

        x1 = np.maximum(boxes[:, None, 0], boxes[None, :, 0])
        y1 = np.maximum(boxes[:, None, 1], boxes[None, :, 1])
        x2 = np.minimum(boxes[:, None, 2], boxes[None, :, 2])
        y2 = np.minimum(boxes[:, None, 3], boxes[None, :, 3])
        inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
        union = areas[:, None] + areas[None, :] - inter
        iou = np.where(union > 0, inter / union, 0.0)
        np.fill_diagonal(iou, 0.0)
        ov_max = iou.max(1) if N > 1 else np.zeros(N)

        delta = 0.0 if prev_n is None else (N - prev_n) / (prev_n + 1)
        expected = N if ewma is None else ewma
        ratio = N / (expected + 1.0)
        ewma = N if ewma is None else EWMA_ALPHA * N + (1 - EWMA_ALPHA) * ewma
        prev_n = N

        for j, r in enumerate(fsub.itertuples()):
            key = (split, seq, int(r.track_id))
            st = state.get(key)
            if st is None:
                st = {'first': frame, 'prev': None, 'n': 0, 'conf': [], 'cen': [], 'lap': None}
                state[key] = st

            st['n'] += 1
            age = frame - st['first'] + 1
            tsu = 0 if st['prev'] is None else max(0, frame - st['prev'] - 1)
            hit = min(1.0, st['n'] / max(age, 1))

            conf = float(r.confidence)
            c5 = st['conf'][-(CONF_WINDOW - 1):] + [conf] if CONF_WINDOW > 1 else [conf]
            conf_std5 = float(np.std(c5)) if len(c5) > 1 else 0.0
            conf_delta = 0.0 if not st['conf'] else conf - st['conf'][-1]

            bw = max(boxes[j, 2] - boxes[j, 0], 1e-6)
            bh = max(boxes[j, 3] - boxes[j, 1], 1e-6)
            min_side = min(bw, bh)
            aspect = abs(bw / bh - r_ref)
            de = float(max(0.0, min(boxes[j, 0], boxes[j, 1], iw - boxes[j, 2], ih - boxes[j, 3]) / D))
            trunc = 1.0 if min(boxes[j, 0], boxes[j, 1], iw - boxes[j, 2], ih - boxes[j, 3]) <= TRUNC_MARGIN_PX else 0.0

            speed = 0.0
            if st['prev'] is not None:
                speed = float(np.hypot(cx[j] - st['cen'][-1][0], cy[j] - st['cen'][-1][1]) / D)

            cen_win = st['cen'][-(PRED_UNCERT_WINDOW - 1):] + [(cx[j], cy[j])] if PRED_UNCERT_WINDOW > 1 else [(cx[j], cy[j])]
            unc = 0.0
            if len(cen_win) > 1:
                arr = np.array(cen_win)
                unc = float(np.sqrt(arr[:, 0].var() + arr[:, 1].var()) / D)

            crop = crop_gray(img, (boxes[j, 0], boxes[j, 1], boxes[j, 2], boxes[j, 3]))
            lv = lap_var(crop)
            lap_log = float(math.log1p(lv))
            bright = float(crop.mean() / 255.0) if crop is not None and crop.size else 0.0
            blur = 0.0
            if st['lap'] is not None and (st['lap'] + lv) > 0:
                blur = float((st['lap'] - lv) / (st['lap'] + lv + 1e-6))

            tstate = 0.0 if age == 1 else (1.0 if tsu == 0 else 2.0)

            rows.append({
                'split': split, 'sequence': seq, 'frame': frame, 'track_id': int(r.track_id),
                'conf': conf, 'conf_std5': conf_std5, 'conf_delta': conf_delta,
                'track_age': float(age), 'hit_ratio': float(hit),
                'time_since_update': float(tsu), 'track_state': tstate,
                'min_side_px': float(min_side), 'aspect_dev': float(aspect),
                'dist_edge': de, 'is_trunc': trunc,
                'speed_norm': speed, 'pred_uncert': unc,
                'lap_var_log': lap_log, 'brightness': bright, 'motion_blur_ratio': blur,
                'nearest_dist_norm': float(nearest[j]),
                'n_neighbors_r2_norm': float(n_nb[j]),
                'overlap_max': float(ov_max[j]),
                'frame_det_delta': float(delta),
                'n_det_expected_ratio': float(ratio),
            })

            st['prev'] = frame
            st['conf'].append(conf)
            st['cen'].append((float(cx[j]), float(cy[j])))
            st['lap'] = lv

df_feat = repair_tracker_features(pd.DataFrame(rows))
df_feat.to_csv(FEATURES_CSV, index=False)
print('features_frame.csv:', len(df_feat), 'dòng |', len(FEATURES), 'đặc trưng')

schema = {
    'features': FEATURES,
    'tracks_source': _TRK_SRC.name,
    'tracks_complete': bool(_TRK_SRC == TRACKS_CSV),
    'count': len(FEATURES),
    'window': {'size': WINDOW_SIZE, 'stride': WINDOW_STRIDE, 'min_frames': MIN_WINDOW_FRAMES},
    'aggregation': {f: ('last' if f in ('conf', 'track_age', 'time_since_update', 'track_state', 'n_det_expected_ratio') else 'mean') for f in FEATURES},
    'definitions': {
        'conf': 'confidence của detection tại frame t',
        'conf_std5': 'std của confidence trong CONF_WINDOW frame gần nhất',
        'conf_delta': 'conf_t - conf_{t-1}',
        'track_age': 't - frame đầu tiên của track + 1',
        'hit_ratio': 'số frame track xuất hiện / track_age',
        'time_since_update': 'số frame trống kể từ lần xuất hiện trước',
        'track_state': '0 tentative, 1 tracked, 2 lost (tái dựng từ chuỗi output ByteTrack)',
        'min_side_px': 'min(w, h) theo pixel',
        'aspect_dev': 'abs(w/h - r_ref), r_ref = median w/h GT train',
        'dist_edge': 'khoảng cách chuẩn hoá tới biên ảnh gần nhất',
        'is_trunc': '1 nếu box chạm biên trong TRUNC_MARGIN_PX',
        'speed_norm': 'dịch chuyển tâm box / đường chéo ảnh',
        'pred_uncert': 'PROXY: std(center) trên PRED_UNCERT_WINDOW frame / D (ByteTrack không expose covariance)',
        'lap_var_log': 'log1p(Var(Laplacian(crop gray)))',
        'brightness': 'mean(crop gray) / 255',
        'motion_blur_ratio': '(L_{t-1} - L_t) / (L_{t-1} + L_t + eps)',
        'nearest_dist_norm': 'khoảng cách tới tâm gần nhất / D',
        'n_neighbors_r2_norm': 'min(số tâm trong bán kính r / K, 1)',
        'overlap_max': 'IoU lớn nhất với box khác trong frame',
        'frame_det_delta': '(N_t - N_{t-1}) / (N_{t-1} + 1)',
        'n_det_expected_ratio': 'N_t / (EWMA(N) + 1), EWMA chỉ dùng quá khứ',
    },
    'notes': [
        'pred_uncert là proxy, không phải trace covariance Kalman.',
        'track_state được tái dựng vì Ultralytics không trả state theo track.',
        'r_ref tính từ GT train, không hard-code.',
    ],
}
SCHEMA_JSON.write_text(json.dumps(schema, indent=2, ensure_ascii=False), encoding='utf-8')
print('feature_schema.json ->', SCHEMA_JSON)
print(df_feat[FEATURES].describe().T[['mean', 'std', 'min', 'max']].round(4).to_string())


!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
CHƯA TRACK XONG: dùng tracks_frame.partial.csv (dữ liệu MỘT PHẦN).
Chạy lại Cell 7/13 để hoàn tất, rồi chạy lại các cell sau.
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
r_ref (median w/h train): 0.5
features_frame.csv: 154243 dòng | 21 đặc trưng
feature_schema.json -> /kaggle/working/artifacts/feature_schema.json
                         mean       std     min       max
conf                   0.6039    0.1014  0.2502    0.8027
conf_std5              0.0270    0.0253  0.0000    0.2125
conf_delta            -0.0015    0.0519 -0.3940    0.4082
track_age             99.3201  117.7428  1.0000  787.0000
hit_ratio              0.9707    0.0917  0.0232    1.0000
time_since_update      0.0820    2.4003  0.0000  448.0000
track_state            0.9941    0.1564  0.0000    2.0000
min_side_px           35.5425   16.9477  6.4300  140.6500
aspect_dev             0.1054    0.1110  0.0000 

In [8]:
# ── CELL 9/13 · Ẩn danh + tấn công Re-ID độc lập ───────────
# Vào : tracks_frame.csv, gt_boxes.csv, sequence_index.csv
# Ra  : samples.csv, attack_results.csv, track_identity.csv

import torch
import torch.nn.functional as F
from torchvision import transforms


def track_identity(df_trk, df_gt, iou_th):
    gt_map = defaultdict(list)
    for r in df_gt.itertuples():
        gt_map[(r.split, r.sequence, int(r.frame))].append(
            (int(r.target_id), r.x1, r.y1, r.x2, r.y2))
    votes = defaultdict(Counter)
    for (split, seq, frame), sub in df_trk.groupby(['split', 'sequence', 'frame']):
        gts = gt_map.get((split, seq, int(frame)))
        if not gts:
            continue
        tb = sub[['x1', 'y1', 'x2', 'y2']].to_numpy(float)
        gb = np.array([g[1:] for g in gts], float)
        ids = [g[0] for g in gts]
        m = xyxy_iou(tb, gb)
        used_t, used_g = set(), set()
        for flat in np.argsort(-m, axis=None):
            i, j = divmod(int(flat), m.shape[1])
            if m[i, j] < iou_th:
                break
            if i in used_t or j in used_g:
                continue
            used_t.add(i)
            used_g.add(j)
            votes[(split, seq, int(sub.iloc[i].track_id))][ids[j]] += 1

    out = {}
    for k, c in votes.items():
        best, n = c.most_common(1)[0]
        out[k] = (int(best), float(n / sum(c.values())))
    return out


def build_samples(df_trk, ident):
    samples = []
    for (split, seq, tid), sub in df_trk.groupby(['split', 'sequence', 'track_id']):
        frames = np.sort(sub.frame.unique())
        if len(frames) < MIN_WINDOW_FRAMES:
            continue
        if WINDOW_SIZE <= 0:
            wins = [(int(frames[0]), int(frames[-1]))]
        else:
            wins = []
            for s in frames[::WINDOW_STRIDE]:
                e = int(s) + WINDOW_SIZE - 1
                if len(frames[(frames >= s) & (frames <= e)]) < MIN_WINDOW_FRAMES:
                    continue
                wins.append((int(s), e))
                if e >= frames[-1]:
                    break
        info = ident.get((split, seq, int(tid)))
        for s, e in wins:
            samples.append({
                'sample_id': f'{split}|{seq}|{tid}|{s}',
                'split': split, 'sequence': seq, 'track_id': int(tid),
                'window_start': s, 'window_end': e,
                'identity': np.nan if info is None else info[0],
                'identity_match_rate': 0.0 if info is None else info[1],
            })
    return pd.DataFrame(samples)


def build_reid():
    device = f'cuda:{DEVICE_EVAL}' if torch.cuda.is_available() else 'cpu'
    model, name = None, None
    if REID_BACKBONE in ('auto', 'osnet_x1_0'):
        try:
            import torchreid
            model = torchreid.models.build_model(name='osnet_x1_0', num_classes=1000, pretrained=True)
            model.classifier = torch.nn.Identity()
            name = 'osnet_x1_0'
        except Exception as e:
            print('OSNet không khả dụng:', str(e)[:140])
            model = None
    if model is None:
        from torchvision.models import resnet50, ResNet50_Weights
        model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        model.fc = torch.nn.Identity()
        name = 'resnet50'
    model.eval().to(device)
    tf = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((REID_INPUT, REID_INPUT)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    print('Re-ID backbone:', name, '| device:', device)
    return model, tf, device, name


def anonymize(img):
    if ANON_METHOD == 'pixelate':
        h, w = img.shape[:2]
        b = max(1, ANON_PIXEL_BLOCK)
        small = cv2.resize(img, (max(1, w // b), max(1, h // b)), interpolation=cv2.INTER_LINEAR)
        return cv2.resize(small, (w, h), interpolation=cv2.INTER_NEAREST)
    k = ANON_KSIZE if ANON_KSIZE % 2 == 1 else ANON_KSIZE + 1
    return cv2.GaussianBlur(img, (k, k), 0)


def crop_color(img, box, pad=0.1):
    if img is None:
        return None
    h, w = img.shape[:2]
    x1, y1, x2, y2 = box
    bw, bh = max(x2 - x1, 1e-6), max(y2 - y1, 1e-6)
    x1 = int(max(0, math.floor(x1 - pad * bw)))
    y1 = int(max(0, math.floor(y1 - pad * bh)))
    x2 = int(min(w, math.ceil(x2 + pad * bw)))
    y2 = int(min(h, math.ceil(y2 + pad * bh)))
    if x2 <= x1 or y2 <= y1:
        return None
    return img[y1:y2, x1:x2]


@torch.no_grad()
def embed_list(model, tf, device, crops):
    if not crops:
        return np.zeros((0, 1), np.float32)
    outs = []
    for i in range(0, len(crops), REID_BATCH):
        batch = torch.stack([tf(c) for c in crops[i:i + REID_BATCH]]).to(device)
        f = F.normalize(model(batch).float(), dim=1)
        outs.append(f.cpu().numpy())
    return np.concatenate(outs, 0)


def build_embedding_cache(df_trk, model, tf, device):
    cache = defaultdict(lambda: {'frames': [], 'clean': [], 'anon': []})
    df_emb = (df_trk.sort_values(['split', 'sequence', 'track_id', 'frame'])
              .assign(_r=lambda d: d.groupby(['split', 'sequence', 'track_id']).cumcount()))
    df_emb = df_emb[df_emb._r % max(1, EMBED_STRIDE) == 0]
    print('crop cần embed:', len(df_emb), '/', len(df_trk))

    for (split, seq, frame), sub in df_emb.groupby(['split', 'sequence', 'frame'], sort=True):
        img = cv2.imread(str(BUILD_DIR / split / 'images' /
                             f'{seq}__{int(frame):07d}.jpg'), cv2.IMREAD_COLOR)
        if img is None:
            continue
        keys, clean, anon = [], [], []
        for r in sub.itertuples():
            c = crop_color(img, (r.x1, r.y1, r.x2, r.y2))
            if c is None or c.size == 0:
                continue
            keys.append((split, seq, int(r.track_id), int(frame)))
            clean.append(c)
            anon.append(anonymize(c))
        if not keys:
            continue
        ec = embed_list(model, tf, device, clean)
        ea = embed_list(model, tf, device, anon)
        for k, a, b in zip(keys, ec, ea):
            d = cache[(k[0], k[1], k[2])]
            d['frames'].append(k[3])
            d['clean'].append(a)
            d['anon'].append(b)

    for k in list(cache):
        d = cache[k]
        order = np.argsort(d['frames'])
        d['frames'] = np.asarray(d['frames'], int)[order]
        d['clean'] = np.stack(d['clean'])[order].astype(np.float32)
        d['anon'] = np.stack(d['anon'])[order].astype(np.float32)
    return dict(cache)


def build_identity_index(cache, ident):
    per = defaultdict(lambda: defaultdict(lambda: {'frames': [], 'embs': []}))
    for (split, seq, tid), d in cache.items():
        info = ident.get((split, seq, tid))
        if info is None or info[1] < LABEL_MIN_MATCH_RATE:
            continue
        slot = per[split][(seq, info[0])]
        slot['frames'].append(d['frames'])
        slot['embs'].append(d['clean'])

    out = {}
    for split, ids in per.items():
        out[split] = {}
        for key, v in ids.items():
            fr = np.concatenate(v['frames'])
            em = np.concatenate(v['embs'])
            order = np.argsort(fr)
            fr, em = fr[order], em[order]
            out[split][key] = {'frames': fr, 'cum': np.cumsum(em, 0), 'n': len(fr)}
    return out


def _unit(v):
    n = float(np.linalg.norm(v))
    return v / n if n > 0 else None


def far_mean(entry, s, e, gap):
    fr, n = entry['frames'], entry['n']
    lo = int(np.searchsorted(fr, s - gap))
    hi = int(np.searchsorted(fr, e + gap + 1))
    n_in = hi - lo
    if n - n_in <= 0:
        return None
    total = entry['cum'][-1]
    insum = entry['cum'][hi - 1] - (entry['cum'][lo - 1] if lo > 0 else 0) if hi > lo else 0
    return _unit((total - insum) / (n - n_in))


def mean_all(entry):
    return _unit(entry['cum'][-1] / entry['n'])


# ── đọc tracks: Cell 7 có thể đã dừng an toàn -> dùng *.partial.csv ──
_TRK_SRC = TRACKS_CSV
if not _TRK_SRC.is_file():
    _TRK_ALT = TRACKS_CSV.with_name(TRACKS_CSV.stem + '.partial.csv')
    if not _TRK_ALT.is_file():
        raise FileNotFoundError(
            f'Không thấy {TRACKS_CSV.name} lẫn {_TRK_ALT.name}. '
            f'Chạy Cell 7/13 trước.')
    _TRK_SRC = _TRK_ALT
    print('!' * 78)
    print(f'CHƯA TRACK XONG: dùng {_TRK_ALT.name} (dữ liệu MỘT PHẦN).')
    print('Chạy lại Cell 7/13 để hoàn tất, rồi chạy lại các cell sau.')
    print('!' * 78)
df_trk = pd.read_csv(_TRK_SRC)
df_gt = pd.read_csv(GT_CSV)

IDENT = track_identity(df_trk, df_gt, TRACK_ID_IOU)
df_tid = pd.DataFrame([{'split': k[0], 'sequence': k[1], 'track_id': k[2],
                        'identity': v[0], 'match_rate': v[1]} for k, v in IDENT.items()])
df_tid.to_csv(ARTIFACT / 'track_identity.csv', index=False)

df_samples = build_samples(df_trk, IDENT)
df_samples.to_csv(SAMPLES_CSV, index=False)
print('samples:', len(df_samples), '| có identity:',
      int(df_samples.identity.notna().sum()))

model, tf, device, reid_name = build_reid()
cache = build_embedding_cache(df_trk, model, tf, device)
print('crop đã embed:', len(cache), 'track')
id_index = build_identity_index(cache, IDENT)
print('identity index:', {k: len(v) for k, v in id_index.items()})

rng = np.random.RandomState(SEED)
attack = []
for r in df_samples.itertuples():
    rec = {'sample_id': r.sample_id, 'split': r.split, 'sequence': r.sequence,
           'track_id': int(r.track_id), 'window_start': int(r.window_start),
           'window_end': int(r.window_end), 'identity_true': r.identity,
           'n_probe': 0, 'n_gallery_ids': 0, 'top1_identity': np.nan,
           'top1_sim': np.nan, 'true_sim': np.nan, 'rank_true': np.nan,
           'top1_correct': np.nan, 'status': 'ok'}

    if pd.isna(r.identity) or r.identity_match_rate < LABEL_MIN_MATCH_RATE:
        rec['status'] = 'no_identity'
        attack.append(rec)
        continue

    d = cache.get((r.split, r.sequence, int(r.track_id)))
    if d is None:
        rec['status'] = 'no_probe'
        attack.append(rec)
        continue

    sel = np.where((d['frames'] >= r.window_start) & (d['frames'] <= r.window_end))[0]
    if len(sel) == 0:
        rec['status'] = 'no_probe'
        attack.append(rec)
        continue
    sel = sel[::PROBE_STRIDE][:MAX_PROBE_CROPS]
    probe = _unit(d['anon'][sel].mean(0))
    if probe is None:
        rec['status'] = 'no_probe'
        attack.append(rec)
        continue
    rec['n_probe'] = int(len(sel))

    candidates = id_index.get(r.split, {})
    keys = list(candidates.keys())
    if GALLERY_CROSS_SEQUENCE:
        pool = keys
    else:
        pool = [k for k in keys if k[0] == r.sequence]
    true_key = (r.sequence, int(r.identity))
    if true_key not in pool:
        rec['status'] = 'no_gallery'
        attack.append(rec)
        continue
    if len(pool) > MAX_GALLERY_IDS:
        others = [k for k in pool if k != true_key]
        pick = rng.choice(len(others), size=min(len(others), MAX_GALLERY_IDS - 1), replace=False)
        pool = [true_key] + [others[i] for i in sorted(pick)]

    sims = {}
    for key in pool:
        entry = candidates[key]
        if key == true_key:
            g = far_mean(entry, r.window_start, r.window_end, GALLERY_MIN_GAP)
        else:
            g = mean_all(entry)
        if g is not None:
            sims[key] = float(probe @ g)

    if true_key not in sims:
        rec['status'] = 'no_gallery'
        attack.append(rec)
        continue
    if len(sims) < MIN_GALLERY_IDS:
        rec['status'] = 'too_few_ids'
        attack.append(rec)
        continue

    order = sorted(sims.items(), key=lambda kv: -kv[1])
    top1_key, top1_sim = order[0]
    rec['n_gallery_ids'] = len(sims)
    rec['top1_identity'] = top1_key[1]
    rec['top1_sim'] = top1_sim
    rec['true_sim'] = sims[true_key]
    rec['rank_true'] = float([k for k, _ in order].index(true_key) + 1)
    rec['top1_correct'] = int(top1_key == true_key)
    attack.append(rec)

df_att = pd.DataFrame(attack)
df_att.to_csv(ATTACK_CSV, index=False)

print('\nattack_results.csv:', len(df_att), 'dòng')
print(df_att.status.value_counts().to_string())
ok = df_att[df_att.status == 'ok']
if len(ok):
    print('top-1 đúng:', f'{ok.top1_correct.mean():.3f}',
          '| sim trung bình:', f'{ok.top1_sim.mean():.3f}')

(ARTIFACT / 'attack_meta.json').write_text(json.dumps({
    'backbone': reid_name, 'anon_method': ANON_METHOD,
    'anon_ksize': ANON_KSIZE, 'anon_block': ANON_PIXEL_BLOCK,
    'gallery_min_gap': GALLERY_MIN_GAP, 'cross_sequence': GALLERY_CROSS_SEQUENCE,
    'window': [WINDOW_SIZE, WINDOW_STRIDE], 'features': len(FEATURES),
}, indent=2), encoding='utf-8')

del model
gc.collect()
torch.cuda.empty_cache()


!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
CHƯA TRACK XONG: dùng tracks_frame.partial.csv (dữ liệu MỘT PHẦN).
Chạy lại Cell 7/13 để hoàn tất, rồi chạy lại các cell sau.
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
samples: 70094 | có identity: 69941


Downloading...
From: https://drive.google.com/uc?id=1LaG1EJpHrxdAxKnSCJ_i0u-nbxSAeiFY
To: /root/.cache/torch/checkpoints/osnet_x1_0_imagenet.pth
100%|██████████| 10.9M/10.9M [00:00<00:00, 234MB/s]


Successfully loaded imagenet pretrained weights from "/root/.cache/torch/checkpoints/osnet_x1_0_imagenet.pth"
Re-ID backbone: osnet_x1_0 | device: cuda:0
crop cần embed: 31783 / 154243
crop đã embed: 2292 track
identity index: {'test': 431, 'train': 887}

attack_results.csv: 70094 dòng
status
ok             61776
no_gallery      4932
no_identity     3386
top-1 đúng: 0.045 | sim trung bình: 0.825


In [12]:
# ── CELL 10/13 · Chọn ngưỡng τ trên val (Youden J) ─────────
# Vào : attack_results.csv
# Ra  : tau.json, roc.png

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

df_att = pd.read_csv(ATTACK_CSV)
val = df_att[(df_att.split == TAU_SELECT_SPLIT) & (df_att.status == 'ok')].copy()
val_classes = sorted(val.top1_correct.dropna().astype(int).unique().tolist()) if len(val) else []
print('mẫu val dùng chọn τ:', len(val), '| lớp:', val_classes)
calibration_split = TAU_SELECT_SPLIT
tau_fallback_split = globals().get('TAU_FALLBACK_SPLIT', 'train')
calibration = val
if len(val_classes) < 2:
    fallback = df_att[(df_att.split == tau_fallback_split) & (df_att.status == 'ok')].copy()
    fallback_classes = sorted(fallback.top1_correct.dropna().astype(int).unique().tolist()) if len(fallback) else []
    print('CẢNH BÁO: val không có đủ hai lớp; fallback classes:', fallback_classes)
    if len(fallback_classes) >= 2:
        calibration_split = tau_fallback_split
        calibration = fallback
    else:
        raise RuntimeError(
            'Không thể hiệu chuẩn tau: cả val và fallback đều thiếu hai lớp. '
            'Cần tăng mẫu attack hoặc sửa cách chia sequence trước khi chạy Cell 10.'
        )
calibration_classes = sorted(calibration.top1_correct.dropna().astype(int).unique().tolist())

tau_info = {'tau': 0.5, 'youden_j': None, 'auc': None,
            'n_val': int(len(val)), 'split': TAU_SELECT_SPLIT,
            'calibration_split': calibration_split,
            'n_calibration': int(len(calibration)),
            'calibration_classes': calibration_classes}

if len(calibration) and calibration.top1_correct.nunique() > 1:
    y = calibration.top1_correct.astype(int).values
    score = calibration.top1_sim.values
    fpr, tpr, thr = roc_curve(y, score)
    j = tpr - fpr
    k = int(np.argmax(j))
    tau = float(np.clip(thr[k], 0.0, 1.0))
    if not np.isfinite(tau):
        tau = 0.5
    tau_info.update({'tau': tau, 'youden_j': float(j[k]),
                     'auc': float(roc_auc_score(y, score)),
                     'tpr': float(tpr[k]), 'fpr': float(fpr[k]),
                     'calibration_split': calibration_split,
                     'n_calibration': int(len(calibration)),
                     'calibration_classes': calibration_classes})

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot(fpr, tpr, label=f'AUC = {tau_info["auc"]:.3f}')
    ax.plot([0, 1], [0, 1], '--', color='gray', lw=1)
    ax.scatter([fpr[k]], [tpr[k]], color='red', zorder=3,
               label=f'τ = {tau:.3f} (J = {j[k]:.3f})')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'Re-ID attack ROC trên {calibration_split}')
    ax.legend(loc='lower right')
    fig.tight_layout()
    fig.savefig(ARTIFACT / 'roc.png', dpi=140)
    plt.close(fig)
else:
    raise RuntimeError('Calibration split must contain both attack classes; no arbitrary tau fallback is allowed')

TAU_JSON.write_text(json.dumps(tau_info, indent=2, ensure_ascii=False), encoding='utf-8')
TAU = float(tau_info['tau'])
print(json.dumps(tau_info, indent=2, ensure_ascii=False))


mẫu val dùng chọn τ: 0 | lớp: []
CẢNH BÁO: val không có đủ hai lớp; fallback classes: [0, 1]
{
  "tau": 0.6931848526000977,
  "youden_j": 0.005750613066240917,
  "auc": 0.33807947387698895,
  "n_val": 0,
  "split": "val",
  "calibration_split": "train",
  "n_calibration": 45001,
  "calibration_classes": [
    0,
    1
  ],
  "tpr": 0.9981217129977461,
  "fpr": 0.9923710999315052
}


In [13]:
# ── CELL 11/13 · Gán nhãn + gộp track-level → risk_dataset.csv ──
# Vào : features_frame.csv, samples.csv, attack_results.csv, tau.json
# Ra  : risk_dataset.csv (21 đặc trưng + label)
# SỬA: CỬA SỔ ĐẶC TRUNG PHẢI CẢUSAL (không dùng frame > window_start / cutoff).

import math
df_feat = pd.read_csv(FEATURES_CSV)
df_samples = pd.read_csv(SAMPLES_CSV)
df_att = pd.read_csv(ATTACK_CSV)
TAU = float(json.loads(TAU_JSON.read_text(encoding="utf-8"))["tau"])

AGG_LAST = {"conf", "track_age", "time_since_update", "track_state", "n_det_expected_ratio"}
att_map = {r.sample_id: r for r in df_att.itertuples()}

feat_groups = {k: g.sort_values("frame")
               for k, g in df_feat.groupby(["split", "sequence", "track_id"])}

records, skipped = [], Counter()
for r in df_samples.itertuples():
    g = feat_groups.get((r.split, r.sequence, int(r.track_id)))
    if g is None:
        skipped["no_features"] += 1
        continue

    # ---- CẢUSAL: chỉ dùng frame <= window_start (không bao giờ nhìn tương lai) ----
    cutoff = int(r.window_start)
    sub = g[g.frame <= cutoff]
    if len(sub) < MIN_WINDOW_FRAMES:
        skipped["short_window"] += 1
        continue

    rec = {
        "sample_id": r.sample_id, "split": r.split, "sequence": r.sequence,
        "track_id": int(r.track_id), "window_start": int(r.window_start),
        "window_end": int(r.window_end), "cutoff": cutoff, "identity": r.identity,
        "n_frames": int(len(sub)),
    }
    for f in FEATURES:
        rec[f] = float(sub[f].iloc[-1]) if f in AGG_LAST else float(sub[f].mean())

    a = att_map.get(r.sample_id)
    if a is None or a.status != "ok" or pd.isna(a.identity_true):
        rec["label"] = np.nan
        rec["label_reason"] = "unreliable" if a is None else a.status
    else:
        rec["top1_identity"] = a.top1_identity
        rec["top1_sim"] = a.top1_sim
        rec["true_sim"] = a.true_sim
        rec["rank_true"] = a.rank_true
        rec["label_reason"] = "attack"
        if int(a.top1_correct) == 1 and a.top1_sim >= TAU:
            rec["label"] = 1.0
        else:
            rec["label"] = 0.0
    records.append(rec)

df_risk = pd.DataFrame(records)
df_risk.to_csv(RISK_CSV, index=False)

print("risk_dataset.csv:", len(df_risk), "mẫu |", len(FEATURES), "đặc trưng")
print("bỏ qua:", dict(skipped))
print("\nphân bố nhãn theo split:")
print(df_risk.groupby("split").label.agg(["count", "sum", "mean"]).round(3).to_string())
print("\nnhãn tổng:", df_risk.label.value_counts(dropna=False).to_dict())
for f in FEATURES:
    assert f in df_risk.columns, f
print("\n21 cột đặc trưng:", FEATURES)


risk_dataset.csv: 70094 mẫu | 21 đặc trưng
bỏ qua: {}

phân bố nhãn theo split:
       count     sum   mean
split                      
test   16775   127.0  0.008
train  45001  2657.0  0.059

nhãn tổng: {0.0: 58992, nan: 8318, 1.0: 2784}

21 cột đặc trưng: ['conf', 'conf_std5', 'conf_delta', 'track_age', 'hit_ratio', 'time_since_update', 'track_state', 'min_side_px', 'aspect_dev', 'dist_edge', 'is_trunc', 'speed_norm', 'pred_uncert', 'lap_var_log', 'brightness', 'motion_blur_ratio', 'nearest_dist_norm', 'n_neighbors_r2_norm', 'overlap_max', 'frame_det_delta', 'n_det_expected_ratio']


In [14]:
# ── CELL 12/13 · Train XGBoost (GroupKFold theo sequence) ──
# Vào : risk_dataset.csv
# Ra  : risk_model.json, risk_model_calibrated.joblib, risk_oof.csv, cv_summary.json

import xgboost as xgb
import joblib
from sklearn.model_selection import GroupKFold
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

df = pd.read_csv(RISK_CSV)
lab = df[df.label.notna()].copy()
lab["label"] = lab.label.astype(int)
print("mẫu có nhãn:", len(lab), "| dương:", int(lab.label.sum()),
      "| âm:", int((1 - lab.label).sum()))
print("sequence:", lab.sequence.nunique())

X = lab[FEATURES].to_numpy(np.float32)
y = lab.label.to_numpy(int)
groups = lab.sequence.to_numpy()
n_groups = len(np.unique(groups))
can_cv = (len(np.unique(y)) > 1) and (n_groups >= 2)
n_splits = max(2, min(N_FOLDS, n_groups)) if can_cv else 0

params = dict(XGB_PARAMS)
if 0 < y.sum() < len(y):
    params["scale_pos_weight"] = float((1 - y.mean()) / max(y.mean(), 1e-6))

oof = np.full(len(lab), np.nan)
fold_models, cv_rows = [], []

if not can_cv:
    print("CẢNH BÁO: chỉ có 1 lớp hoặc <2 sequence → không thể CV, dùng prior làm baseline.")
    oof[:] = float(y.mean())
    cv_summary = {"n_samples": int(len(lab)), "n_groups": int(n_groups),
                  "degenerate": True, "prior": float(y.mean())}
else:
    gkf = GroupKFold(n_splits=n_splits)
    for fold, (tr, va) in enumerate(gkf.split(X, y, groups)):
        model = xgb.XGBClassifier(**params)
        model.fit(X[tr], y[tr], eval_set=[(X[va], y[va])], verbose=False)
        oof[va] = model.predict_proba(X[va])[:, 1]
        auc = roc_auc_score(y[va], oof[va]) if len(np.unique(y[va])) > 1 else np.nan
        ap = average_precision_score(y[va], oof[va]) if len(np.unique(y[va])) > 1 else np.nan
        cv_rows.append({"fold": fold, "n_train": int(len(tr)), "n_val": int(len(va)),
                        "auc": None if np.isnan(auc) else float(auc),
                        "pr_auc": None if np.isnan(ap) else float(ap),
                        "best_iteration": int(getattr(model, "best_iteration", 0) or 0)})
        fold_models.append(model)
        print(f"  fold {fold}: n_tr={len(tr)} n_va={len(va)} AUC={auc:.4f} PR-AUC={ap:.4f}")

    cv_summary = {"n_samples": int(len(lab)), "n_groups": int(n_groups),
                  "n_splits": int(n_splits), "degenerate": False,
                  "folds": cv_rows,
                  "oof_auc": float(roc_auc_score(y, oof)),
                  "oof_pr_auc": float(average_precision_score(y, oof))}

# ---- Huấn luyện model cuối trên toàn bộ dữ liệu có nhãn ----
final_model = xgb.XGBClassifier(**{k: v for k, v in params.items()
                                   if k != "early_stopping_rounds"})
final_model.fit(X, y, verbose=False)
final_model.save_model(str(RISK_MODEL_JSON))
save_model_metadata(RISK_MODEL_JSON, FEATURES)
print("risk_model.json ->", RISK_MODEL_JSON)

# ---- Calibration (Platt scaling / Isotonic) trên OOF ----
if len(np.unique(y)) > 1 and len(lab) > 50:
    # Dùng OOF predictions làm input cho calibration (không rò rỉ vì OOF là out-of-fold)
    cal = CalibratedClassifierCV(final_model, method="isotonic", cv="prefit")
    cal.fit(X, y)
    p_raw = final_model.predict_proba(X)[:, 1]
    p_cal = cal.predict_proba(X)[:, 1]

    brier_raw = float(brier_score_loss(y, p_raw))
    brier_cal = float(brier_score_loss(y, p_cal))
    print(f"Brier trước calibration: {brier_raw:.4f} | sau calibration: {brier_cal:.4f}")

    # Lưu model đã calibration
    joblib.dump(cal, ARTIFACT / "risk_model_calibrated.joblib")
    print("risk_model_calibrated.joblib ->", ARTIFACT / "risk_model_calibrated.joblib")

    cv_summary["brier_raw"] = brier_raw
    cv_summary["brier_calibrated"] = brier_cal
    cv_summary["calibration_method"] = "isotonic_prefit_on_oof"
else:
    cv_summary["calibration_method"] = "skipped_insufficient_samples"

pd.DataFrame({"sample_id": lab.sample_id, "split": lab.split,
              "sequence": lab.sequence, "y_true": y, "y_prob": oof}).to_csv(RISK_OOF_CSV, index=False)
(ARTIFACT / "cv_summary.json").write_text(json.dumps(cv_summary, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps({k: v for k, v in cv_summary.items() if k != "folds"}, indent=2))


mẫu có nhãn: 61776 | dương: 2784 | âm: 58992
sequence: 39
  fold 0: n_tr=47001 n_va=14775 AUC=0.6287 PR-AUC=0.1560
  fold 1: n_tr=50026 n_va=11750 AUC=0.7361 PR-AUC=0.0679
  fold 2: n_tr=50026 n_va=11750 AUC=0.5451 PR-AUC=0.0579
  fold 3: n_tr=50026 n_va=11750 AUC=0.7588 PR-AUC=0.1023
  fold 4: n_tr=50025 n_va=11751 AUC=0.4316 PR-AUC=0.0054
risk_model.json -> /kaggle/working/artifacts/risk_model.json
{
  "n_samples": 61776,
  "n_groups": 39,
  "n_splits": 5,
  "degenerate": false,
  "oof_auc": 0.48092960844193955,
  "oof_pr_auc": 0.048393136014234034
}


In [15]:
# ── CELL 13/13 · Đo hiệu năng + SHAP + export ──────────────
# Vào : risk_model.json, risk_model_calibrated.joblib, risk_oof.csv, risk_dataset.csv
# Ra  : risk_metrics.json, calib.png, importance.png, shap.png, zip

from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                             precision_score, recall_score, brier_score_loss,
                             roc_curve, confusion_matrix)
from sklearn.calibration import calibration_curve, expected_calibration_error
from sklearn.linear_model import LogisticRegression
import joblib
import matplotlib.pyplot as plt

df = pd.read_csv(RISK_CSV)
lab = df[df.label.notna()].copy()
lab["label"] = lab.label.astype(int)
oof = pd.read_csv(RISK_OOF_CSV)
load_model_metadata(RISK_MODEL_JSON)

X = lab[FEATURES].to_numpy(np.float32)
y = lab.label.to_numpy(int)
p_raw = oof.y_prob.to_numpy(float)

# ---- Thử tải model đã calibration; nếu không có, dùng raw ----
cal_path = ARTIFACT / "risk_model_calibrated.joblib"
p_cal = None
cal_used = False
if cal_path.is_file():
    try:
        cal = joblib.load(cal_path)
        p_cal = cal.predict_proba(X)[:, 1]
        cal_used = True
        print("Dùng model đã calibration (isotonic).")
    except Exception as e:
        print("Không load được model calibration:", str(e)[:160])

p = p_cal if cal_used else p_raw

# ---- chọn ngưỡng quyết định trên OUF (calibrated nếu có) ----
thr_info = {"threshold": RISK_THRESHOLD, "method": "fixed",
            "calibrated": bool(cal_used)}
if len(np.unique(y)) > 1:
    fpr, tpr, thr = roc_curve(y, p)
    j = tpr - fpr
    k = int(np.argmax(j))
    thr_info = {"threshold": float(np.clip(thr[k], 0, 1)), "method": "youden_j_oof",
                "j": float(j[k]), "calibrated": bool(cal_used)}
thr = thr_info["threshold"]
y_hat = (p >= thr).astype(int)

def expected_calibration_error(y_true, prob, bins=10):
    edges = np.linspace(0.0, 1.0, bins + 1)
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (prob >= lo) & (prob < hi if hi < 1 else prob <= hi)
        if mask.any():
            ece += float(mask.mean()) * abs(float(prob[mask].mean()) - float(y_true[mask].mean()))
    return ece

metrics = {
    "n_samples": int(len(lab)),
    "n_positive": int(y.sum()),
    "positive_rate": float(y.mean()),
    "threshold": thr_info,
    "auc": float(roc_auc_score(y, p)) if len(np.unique(y)) > 1 else None,
    "pr_auc": float(average_precision_score(y, p)) if len(np.unique(y)) > 1 else None,
    "f1": float(f1_score(y, y_hat, zero_division=0)),
    "precision": float(precision_score(y, y_hat, zero_division=0)),
    "recall": float(recall_score(y, y_hat, zero_division=0)),
    "brier": float(brier_score_loss(y, p)) if len(np.unique(y)) > 1 else None,
    "ece": expected_calibration_error(y, p) if len(np.unique(y)) > 1 else None,
    "calibration_used": bool(cal_used),
    "evaluation_protocol": "sequence-grouped OOF over all labelled sequences; causal features (no look-ahead)",
    "confusion_matrix": confusion_matrix(y, y_hat).tolist(),
}

# ---- baseline so sánh ----
baselines = {}
if len(np.unique(y)) > 1:
    prior = np.full(len(y), y.mean())
    baselines["prior"] = {"brier": float(brier_score_loss(y, prior)),
                          "auc": 0.5}
    lr = LogisticRegression(max_iter=2000, class_weight="balanced")
    lr.fit(X, y)
    lp = lr.predict_proba(X)[:, 1]
    baselines["logreg_in_sample"] = {"auc": float(roc_auc_score(y, lp)),
                                     "brier": float(brier_score_loss(y, lp))}
    conf_idx = FEATURES.index("conf")
    cp = X[:, conf_idx]
    baselines["conf_threshold"] = {"auc": float(roc_auc_score(y, cp))}
    metrics["baselines"] = baselines

# ---- calibration curve ----
if len(np.unique(y)) > 1:
    try:
        frac_pos, mean_pred = calibration_curve(y, p, n_bins=10, strategy="quantile")
        fig, ax = plt.subplots(figsize=(5, 5))
        label = "XGBoost (calibrated OOF)" if cal_used else "XGBoost (raw OOF)"
        ax.plot(mean_pred, frac_pos, "o-", label=label)
        ax.plot([0, 1], [0, 1], "--", color="gray", lw=1)
        ax.set_xlabel("Xác suất dự đoán")
        ax.set_ylabel("Tỷ lệ dương thực tế")
        ax.set_title(f"Calibration | Brier = {metrics['brier']:.4f} | ECE = {metrics['ece']:.4f}")
        ax.legend()
        fig.tight_layout()
        fig.savefig(ARTIFACT / "calib.png", dpi=140)
        plt.close(fig)
    except Exception as e:
        print("Calibration lỗi:", str(e)[:160])

# ---- feature importance ----
model = xgb.XGBClassifier()
model.load_model(str(RISK_MODEL_JSON))
gain_map = model.get_booster().get_score(importance_type="gain")
imp = pd.DataFrame({
    "feature": FEATURES,
    "gain": [float(gain_map.get(f"f{i}", gain_map.get(f, 0.0))) for i, f in enumerate(FEATURES)],
})
imp = imp.sort_values("gain", ascending=False)
imp.to_csv(ARTIFACT / "feature_importance.csv", index=False)

fig, ax = plt.subplots(figsize=(6, 6))
ax.barh(imp.feature[::-1], imp.gain[::-1], color="steelblue")
ax.set_xlabel("Gain")
ax.set_title("XGBoost feature importance")
fig.tight_layout()
fig.savefig(ARTIFACT / "importance.png", dpi=140)
plt.close(fig)

# ---- SHAP (tuỳ chọn) ----
try:
    import shap
    explainer = shap.TreeExplainer(model)
    sample_n = min(500, len(X))
    sv = explainer.shap_values(X[:sample_n])
    fig = plt.figure(figsize=(8, 6))
    shap.summary_plot(sv, X[:sample_n], feature_names=FEATURES, show=False)
    fig.tight_layout()
    fig.savefig(ARTIFACT / "shap.png", dpi=140, bbox_inches="tight")
    plt.close(fig)
    metrics["shap"] = "shap.png"
except Exception as e:
    print("SHAP không khả dụng:", str(e)[:160])
    metrics["shap"] = None

# ---- export detector ----
if EXPORT_ONNX:
    try:
        exp = YOLO(str(BEST_MODEL))
        onnx_path = Path(exp.export(format="onnx", imgsz=IMGSZ_USED, opset=ONNX_OPSET,
                                    dynamic=False, simplify=True, half=False))
        shutil.copy2(onnx_path, ARTIFACT / onnx_path.name)
        print("ONNX ->", ARTIFACT / onnx_path.name)
        del exp
        gc.collect()
        torch.cuda.empty_cache()
    except Exception as e:
        print("Export ONNX lỗi:", str(e)[:160])

shutil.copy2(BEST_MODEL, ARTIFACT / "yolo26s.pt")
shutil.copy2(DATA_YAML, ARTIFACT / "data_person.yaml")

RISK_METRICS_JSON.write_text(json.dumps(metrics, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps({k: v for k, v in metrics.items() if k != "confusion_matrix"},
                 indent=2, ensure_ascii=False))

zip_path = create_results_zip() if "create_results_zip" in globals() else WORK / "visdrone_privacy_results.zip"
print("\nfile ZIP tổng ->", zip_path, f"({Path(zip_path).stat().st_size / 2**20:.1f} MiB)")
print("artifact:", sorted(p.name for p in ARTIFACT.iterdir()))


WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Ultralytics 8.4.197 🚀 Python-3.13.15 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLO26n summary (fused): 121 layers, 2,389,735 parameters, 0 gradients, 14.2 GFLOPs

PyTorch: starting from '/kaggle/input/datasets/mtu2006/1234567/runs/yolo26n_visdrone_person_t4x2/weights/yolo26s.pt' with input shape (1, 3, 1024, 1024) BCHW and output shape(s) (1, 5, 21504) (5.2 MB)
requirements: Ultralytics requirement ['onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 10 packages in 245ms
Prepared 1 package in 46ms
Installed 1 package in 9ms
 + onnxslim==0.1.98

requirements: AutoUpdate success ✅ 0.5s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with